# Данные и результаты эксперимента

RAG-поиск по корпоративной базе знаний «Вектор-Системы».

Этот ноутбук — **выкладка данных** к статье: корпус знаний, банк вопросов и
сырые ответы моделей. Никаких выводов и анализа здесь нет — только данные.

> Все данные **синтетические**: компания «Вектор-Системы» вымышлена, все
> числа, даты и формулировки выдуманы; совпадения с реальными компаниями случайны.

## Состав

| Данные | Что это | Объём |
|---|---|---|
| `CORPUS` | 10 документов регламентов (Markdown), чанки = разделы документа | 10 файлов |
| `QUESTIONS` | Банк вопросов с типом, золотыми документами и эталонными ответами | 18 вопросов |
| `RESULTS` | Ответы моделей: 10 моделей × 18 вопросов × 2 условия (A, B) × 2 прогона | 720 записей |

Модели: 10 (M10-GLM-4.7-Flash-Q6_K, M11-gemma-4-31B-it-Q6_K, M12-T-pro-it-2.1-Q6_K, M13-Qwen3.5-35B-A3B-UD-Q6_K, M6-GigaChat3.1-10B-A1.8B-q8_0, M7-gpt-oss-20b-Q8_0, M8-saiga_gemma3_12b-Q8_0, M9-gemma-4-12b-it-Q8_0, YandexGPT-5-Lite-8B-instruct-Q4_K_M, gemma-7b-it-Q8_0).

## Условия эксперимента

- **A** — в промпт модели подавались 5 отретриванных фрагментов корпуса (BM25, top-5);
- **B** — в промпт подавался только вопрос, фрагменты не подавались.

## Обозначения

- `ДNN#K` (поле `retrieved`) — фрагмент: документ ДNN, раздел K (чанкование — по разделам).
- Типы вопросов: `single` — ответ в одном документе; `multi` — ответ требует нескольких
  документов; `trap` — «ловушка»: данных нет в корпусе, корректный ответ — отказ
  («Данных по этому вопросу в базе знаний нет»); `paraphrase` — переформулировка/отрицание
  факта из корпуса.

## Структура ноутбука

1. **Данные** — сырые исходники (корпус, банк вопросов, JSONL с ответами) в виде Python-констант.
2. **Корпус знаний** — все 10 документов.
3. **Банк вопросов** — таблица из 18 вопросов.
4. **Ответы моделей** — полные таблицы (720 записей) по условиям A и B.


In [ ]:
# Данные: синтетический корпус знаний «Вектор-Системы» (10 документов, Markdown).
# Ключ — имя файла, значение — полный текст документа. Чанк «ДNN#K» = раздел K документа ДNN.

CORPUS = {
    "D01-oformlenie-sotrudnikov.md": r"""# Д01. Регламент оформления сотрудников ООО «Вектор-Системы»

Введён 15.01.2025. Область: все новые сотрудники компании.

## 1. Процедура приёма
1.1. Приём оформляется после оффера HR и заявки руководителя подразделения.
1.2. Срок испытательного срока — 3 месяца; для руководителей подразделений — 6 месяцев.
1.3. В первый рабочий день сотруднику назначается наставник из того же подразделения.
1.4. Испытательный период (проверочное задание с оценкой) проводится в первые 2 недели работы.

## 2. Документы
2.1. Трудовой договор подписывается в первый рабочий день до 12:00.
2.2. Пропуск оформляется службой безопасности в течение 2 рабочих дней после подписания договора.
2.3. Копии документов передаются в кадровый отдел через СЭД «Дирекум».

## 3. Оценка испытательного срока
3.1. Наставник и руководитель дают письменную оценку за 10 дней до конца испытательного срока.
3.2. При отрицательной оценке срок может быть продлён один раз на 1 месяц.
3.3. Результат фиксируется в СЭД «Дирекум» и влияет на допуск к проектным работам.""",
    "D02-informatsionnaya-bezopasnost.md": r"""# Д02. Политика информационной безопасности ООО «Вектор-Системы»

Введён 03.02.2025. Область: все сотрудники, подрядчики с доступом к системам.

## 1. Пароли и учётные записи
1.1. Минимальная длина пароля — 12 знаков, обязательны цифры и спецсимволы.
1.2. Ротация пароля — каждые 90 дней.
1.3. Запрещено передавать пароль по почте, в мессенджерах и устно.
1.4. Учётная запись подрядчика закрывается в течение 1 рабочего дня после окончания договора.

## 2. Удалённый доступ
2.1. Доступ к внутренним системам вне офиса — только через корпоративный VPN.
2.2. Публичные Wi-Fi сети (кафе, аэропорты) для работы с клиентскими данными запрещены.

## 3. Данные клиентов
3.1. Пересылка клиентских данных на личную почту запрещена.
3.2. Обмен клиентскими файлами — только через корпоративное хранилище с правами доступа.
3.3. Инциденты утечки сообщаются в службу безопасности по телефону 300 или почте sec@vector-sys.example.""",
    "D03-komandirovochnaya-politika.md": r"""# Д03. Командировочная политика ООО «Вектор-Системы»

Введён 20.03.2025. Область: служебные поездки сотрудников.

## 1. Суточные и лимиты
1.1. Суточные — 1500 рублей за каждый день поездки, включая дни в пути.
1.2. Гостиница — не дороже 6500 рублей за ночь; для городов присутствия клиента — лимит 8000 рублей по согласованию с руководителем.
1.3. Железнодорожный билет — купе по умолчанию, плацкарт запрещён.

## 2. Оформление
2.1. Поездка оформляется заявкой в СЭД «Дирекум» не позднее чем за 5 рабочих дней до выезда.
2.2. Аванс на поездку выдаётся в течение 2 рабочих дней после согласования заявки.

## 3. Отчётность
3.1. Авансовый отчёт сдаётся в течение 3 рабочих дней после возвращения.
3.2. К отчёту прилагаются чеки; чеки без реквизитов отеля не принимаются.
3.3. Перерасход более 10 % от лимита возмещается только с письменной визы руководителя.""",
    "D04-udalennaya-rabota.md": r"""# Д04. Регламент удалённой работы ООО «Вектор-Системы»

Введён 10.04.2025. Область: сотрудники, переведённые на гибридный формат.

## 1. Формат
1.1. Гибридный график: не менее 2 рабочих дней в офисе в неделю (дни определяет руководитель).
1.2. Рабочий график: 9:00–18:00, перерыв 13:00–14:00.
1.3. Ядро доступности — 11:00–15:00: сотрудник обязан быть доступен по видео и телефону.

## 2. Оборудование и связь
2.1. Рабочий ноутбук и гарнитура выдаются компанией; личное оборудование для клиентских данных не используется.
2.2. Резервный канал связи (мобильный интернет) обязателен при работе с критичными проектами.

## 3. Перевод на удалёнку
3.1. Перевод оформляется заявкой в СЭД «Дирекум» с визой руководителя, срок рассмотрения — 3 рабочих дня.
3.2. Удалённый сотрудник проходит инструктаж по безопасности (Д02) ежегодно.
3.3. Перевод может быть отозван по требованию проекта с уведомлением за 5 рабочих дней.""",
    "D05-oborudovanie.md": r"""# Д05. Выдача и возврат оборудования ООО «Вектор-Системы»

Введён 12.05.2025. Область: ИТ-оборудование сотрудников.

## 1. Сроки обновления
1.1. Рабочий ноутбук обновляется каждые 3 года с даты выдачи.
1.2. Монитор обновляется каждые 5 лет; второй монитор выдаётся по заявке руководителя.
1.3. Периферия (клавиатура, мышь, гарнитура) — по заявке при выходе из строя.

## 2. Заявки
2.1. Заявка на оборудование подаётся через сервис HelpDesk, категория «Оборудование».
2.2. Срок исполнения заявки — 5 рабочих дней; для критичных проектов — 1 рабочий день.

## 3. Возврат
3.1. При увольнении оборудование возвращается в течение 10 календарных дней с последней рабочей даты.
3.2. За выявленные повреждения не по вине сотрудника ремонт не удерживается.
3.3. Незакрытые заявки на оборудование блокируют финальный расчёт кадрового отдела.""",
    "D06-sla-podderzhki.md": r"""# Д06. Внутренний SLA службы поддержки ООО «Вектор-Системы»

Введён 01.06.2025. Область: внутренние заявки сотрудников в службу поддержки.

## 1. Категории заявок
1.1. Critical — полный отказ сервиса, работа подразделения остановлена.
1.2. Major — сервис работает частично, затронуты не все пользователи.
1.3. Minor — деградация, есть обходной путь.

## 2. Время реакции и решения
2.1. Critical: реакция — 30 минут, решение — 4 часа.
2.2. Major: реакция — 2 часа, решение — 1 рабочий день.
2.3. Minor: реакция — 8 часов, решение — 3 рабочих дня.

## 3. Эскалация
3.1. При превышении времени реакции Critical заявка эскалируется руководителю службы поддержки автоматически.
3.2. Превышение времени решения Major фиксируется в ежемесячном отчёте качества поддержки.
3.3. Заявки, созданные сотрудником с категорией ниже фактической, переклассифицируются поддержкой с уведомлением автора.""",
    "D07-klassifikatsiya-incidentov.md": r"""# Д07. Классификация инцидентов ООО «Вектор-Системы»

Введён 01.06.2025. Область: инциденты на системах клиентов и внутренних сервисах.

## 1. Классы инцидентов
1.1. Класс A — полный отказ сервиса клиента, обслуживание невозможно.
1.2. Класс B — частичный отказ: часть функций недоступна части пользователей.
1.3. Класс C — деградация производительности при сохранении функций.

## 2. Уведомление
2.1. Инцидент класса A уведомляется на горячую линию в течение 15 минут с момента обнаружения.
2.2. Класс B — в течение 1 часа. Класс C — в течение 1 рабочего дня.
2.3. Уведомление клиента о классе A делает руководитель проекта, о классах B и C — инженер поддержки.

## 3. Постмортем
3.1. По инциденту класса A постмортем проводится в течение 5 рабочих дней после восстановления.
3.2. По классу B постмортем обязателен при повторе в течение месяца.
3.3. Отчёт постмортема публикуется в базе знаний с префиксом «PM-».""",
    "D08-prays-vektor-ofis.md": r"""# Д08. Прайс-лист линейки «Вектор-Офис» (редакция 2026)

Введён 10.01.2026. Область: тарифы продукта «Вектор-Офис» для клиентов.

## 1. Тарифы (за пользователя в месяц)
1.1. «Вектор-Офис Basic» — 1200 рублей: учёт обращений, базовые отчёты.
1.2. «Вектор-Офис Standard» — 2400 рублей: SLA-маршрутизация, интеграции, расширенные отчёты.
1.3. «Вектор-Офис Enterprise» — 4900 рублей: выделенный контур, персональный менеджер, SLA 99,9 %.

## 2. Скидки и условия
2.1. При годовой оплате — скидка 10 % от тарифа.
2.2. Для клиентов от 500 пользователей — дополнительная скидка 5 %, скидки суммируются.
2.3. Тестовый период — 30 дней, карта не требуется.

## 3. Оплата
3.1. Счёт выставляется 1-го числа месяца, оплата — 10 рабочих дней.
3.2. Просрочка более 20 дней ведёт к блокировке контура с уведомлением за 3 рабочих дня.""",
    "D09-otpusknoy-reglament.md": r"""# Д09. Отпускной регламент ООО «Вектор-Системы»

Введён 25.07.2025. Область: ежегодные оплачиваемые отпуска.

## 1. Продолжительность
1.1. Ежегодный отпуск — 28 календарных дней.
1.2. Дополнительный отпуск за стаж в компании: от 5 лет — 3 дня, от 10 лет — 5 дней.

## 2. Планирование
2.1. График отпусков утверждается до 15 декабря на следующий год.
2.2. Перенос отпуска внутри года — уведомление за 1 месяц, согласование с руководителем.
2.3. Разделение отпуска допускается; минимальная часть — 7 дней, одна из частей — не менее 14 календарных дней.

## 3. Оформление
3.1. Заявление подаётся в СЭД «Дирекум» не позднее чем за 2 недели до начала отпуска.
3.2. Отпускные выплачиваются за 3 календарных дня до начала отпуска.
3.3. Отзыв из отпуска — только приказом генерального директора с письменного согласия сотрудника.""",
    "D10-dokumentoborot-sed.md": r"""# Д10. Документооборот и СЭД «Дирекум» ООО «Вектор-Системы»

Введён 02.09.2025. Область: внутренние документы, договоры, регламенты.

## 1. Регистрация
1.1. Все документы регистрируются в СЭД «Дирекум»; бумажные копии — только для договоров с клиентом.
1.2. Формат регистрационного номера: VS-ГГГГ-NNNNN, где ГГГГ — год, NNNNN — порядковый номер.

## 2. Согласование
2.1. Цикл согласования договора — не более 5 рабочих дней: юрист 2 дня, финансы 2 дня, директор 1 день.
2.2. Регламенты подразделения согласуются 3 рабочих дня и подписываются генеральным директором.
2.3. Замечания согласующего снимаются автором в течение 1 рабочего дня, иначе цикл приостанавливается.

## 3. Хранение
3.1. Срок хранения внутренних регламентов — 5 лет после замены новым изданием.
3.2. Доступ к договорам с клиентами — по ролям «юрист», «руководитель проекта», «финансы».
3.3. Архивное удаление производит администратор СЭД по квартальному плану.""",
}

print(f"Документов: {len(CORPUS)}; всего знаков: {sum(len(v) for v in CORPUS.values())}")

In [ ]:
# Данные: банк вопросов (сырой исходник questions.md) и разобранная структура QUESTIONS.

QUESTIONS_RAW = r"""# Банк вопросов — Самара (партия 1: 18 из 40)

Формат: `QNN | тип | золотой документ(ы) | вопрос | эталон ответа`.
Эталон — единственный проверяемый источник разметки. Ловушки: в корпусе ответа нет → правильный ответ = отказ («Данных по этому вопросу в базе знаний нет»).

## Однодокументные (8)

**Q01** | single | Д01 | Сколько длится испытательный срок обычного сотрудника и для руководителей? | Обычный — 3 месяца, руководители подразделений — 6 месяцев.
**Q02** | single | Д02 | Как часто нужно менять пароль и какая минимальная длина? | Каждые 90 дней; минимум 12 знаков, с цифрами и спецсимволами.
**Q03** | single | Д03 | Какие суточные при командировке и лимит на гостиницу? | Суточные 1500 руб./день (дни в пути включены); отель ≤ 6500 руб./ночь (в городах клиента — до 8000 по согласованию).
**Q04** | single | Д04 | Сколько дней в неделю удалёнщик обязан быть в офисе и каково ядро доступности? | Не менее 2 дней в офисе; ядро доступности 11:00–15:00.
**Q05** | single | Д05 | Как часто обновляются ноутбук и монитор и куда подавать заявку? | Ноутбук — каждые 3 года, монитор — каждые 5 лет; заявка через HelpDesk, категория «Оборудование».
**Q06** | single | Д06 | Каковы время реакции и решения для заявок Critical и Major? | Critical: реакция 30 мин, решение 4 ч; Major: реакция 2 ч, решение 1 рабочий день.
**Q07** | single | Д08 | Сколько стоит тариф «Вектор-Офис Standard» и есть ли скидка за годовую оплату? | 2400 руб./мес за пользователя; скидка 10 % при годовой оплате.
**Q08** | single | Д09 | Сколько дней ежегодного отпуска и за сколько подаётся заявление? | 28 календарных дней; заявление в СЭД «Дирекум» не позднее чем за 2 недели.

## Многодокументные (3)

**Q09** | multi | Д04+Д05 | Удалёнщик хочет второй монитор для клиентских проектов — как получить и что учесть? | Второй монитор — по заявке руководителя (Д05); заявка через HelpDesk, 5 раб. дней (для критичных проектов — 1 день); клиентские данные только на корпоративном оборудовании, через VPN (Д04, Д02).
**Q10** | multi | Д03+Д10 | Сотрудник едет в командировку: когда подавать заявку в СЭД «Дирекум», сколько можно тратить на гостиницу и когда сдавать авансовый отчёт? | Заявка в СЭД «Дирекум» за 5 раб. дней до выезда; отель ≤ 6500 руб./ночь; авансовый отчёт — 3 раб. дня после возвращения.
**Q11** | multi | Д10+Д08 | Клиент просит договор на тариф Enterprise с годовой оплатой: сколько длится цикл согласования договора и какова цена со скидкой за год? | Цикл согласования ≤ 5 раб. дней (юрист 2 + финансы 2 + директор 1); Enterprise 4900 руб./мес/пользователь, годовая оплата −10 % (при ≥500 пользователей ещё −5 %, суммируются).

## Ловушки (5) — правильный ответ: отказ

**Q12** | trap | — | Каков размер годовой премии сотрудников? | В корпусе нет данных → «Данных по этому вопросу в базе знаний нет».
**Q13** | trap | — | Как оформить корпоративную SIM-карту? | Нет данных → отказ.
**Q14** | trap | — | Сколько стоит тариф «Вектор-Офис Premium»? | Тарифа Premium в корпусе нет (только Basic/Standard/Enterprise) → отказ.
**Q15** | trap | — | Какова процедура оформления материнского капитала? | Нет данных → отказ.
**Q16** | trap | — | Где находится столовая в офисе и каково меню? | Нет данных → отказ.

## Парадные (2) — переформулировка/отрицание

**Q17** | paraphrase | Д02 | Верно ли, что пароль нужно менять каждые 60 дней? | Нет: ротация каждые 90 дней.
**Q18** | paraphrase | Д09 | Можно ли разбить весь отпуск на однодневные части? | Нет: минимальная часть 7 дней, одна из частей ≥ 14 календарных дней.

## Баланс партии 2 (добить до 40)

+12 однодокументных (по Д11–Д24), +5 многодокументных, +3 ловушки, +2 парадных (итого 40)."""

import re

QUESTIONS = []
for _line in QUESTIONS_RAW.splitlines():
    _line = _line.strip()
    if not _line.startswith("**Q"):
        continue
    _qid, _qtype, _gold, _qtext, _ref = [x.strip() for x in _line.split(" | ")]
    QUESTIONS.append({"id": _qid.strip("*"), "type": _qtype, "gold": _gold,
                        "question": _qtext, "reference": _ref})

print(f"Вопросов: {len(QUESTIONS)} (single: {sum(q['type']=='single' for q in QUESTIONS)}, "
      f"multi: {sum(q['type']=='multi' for q in QUESTIONS)}, "
      f"trap: {sum(q['type']=='trap' for q in QUESTIONS)}, "
      f"paraphrase: {sum(q['type']=='paraphrase' for q in QUESTIONS)})")

In [ ]:
# Данные: сырые ответы моделей (JSON Lines, 720 записей) и разобранный список RESULTS.

ANSWERS_RAW = r"""{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q01", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 0.96, "prompt_tokens": 615, "completion_tokens": 24, "answer": " Срок испытательного срока — 3 месяца; для руководителей подразделений — 6 месяцев [Д01#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q01", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 0.35, "prompt_tokens": 615, "completion_tokens": 24, "answer": " Срок испытательного срока — 3 месяца; для руководителей подразделений — 6 месяцев [Д01#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q01", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 0.22, "prompt_tokens": 98, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q01", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 0.11, "prompt_tokens": 98, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q02", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.66, "prompt_tokens": 355, "completion_tokens": 25, "answer": " Пароль нужно менять каждые 90 дней, минимальная длина пароля — 12 знаков [Д02#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q02", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.36, "prompt_tokens": 355, "completion_tokens": 25, "answer": " Пароль нужно менять каждые 90 дней, минимальная длина пароля — 12 знаков [Д02#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q02", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.16, "prompt_tokens": 97, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q02", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.11, "prompt_tokens": 97, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q03", "condition": "A", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 1.32, "prompt_tokens": 610, "completion_tokens": 64, "answer": " Суточные — 1500 рублей за каждый день поездки, включая дни в пути [Д03#1]. Лимит на гостиницу — не дороже 6500 рублей за ночь; для городов присутствия клиента — лимит 8000 рублей по согласованию с руководителем [Д03#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q03", "condition": "A", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 0.93, "prompt_tokens": 610, "completion_tokens": 64, "answer": " Суточные — 1500 рублей за каждый день поездки, включая дни в пути [Д03#1]. Лимит на гостиницу — не дороже 6500 рублей за ночь; для городов присутствия клиента — лимит 8000 рублей по согласованию с руководителем [Д03#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q03", "condition": "B", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 0.18, "prompt_tokens": 98, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q03", "condition": "B", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 0.11, "prompt_tokens": 98, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q04", "condition": "A", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 1.24, "prompt_tokens": 650, "completion_tokens": 53, "answer": " Удалёнщик обязан быть в офисе не менее 2 рабочих дней в неделю (дни определяет руководитель) [Д04#1]. Ядро доступности — с 11:00 до 15:00 [Д04#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q04", "condition": "A", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 0.77, "prompt_tokens": 650, "completion_tokens": 53, "answer": " Удалёнщик обязан быть в офисе не менее 2 рабочих дней в неделю (дни определяет руководитель) [Д04#1]. Ядро доступности — с 11:00 до 15:00 [Д04#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q04", "condition": "B", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 0.18, "prompt_tokens": 102, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q04", "condition": "B", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 0.12, "prompt_tokens": 102, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q05", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.82, "prompt_tokens": 317, "completion_tokens": 38, "answer": " Ноутбук обновляется каждые 3 года с даты выдачи [Д05#1]. Монитор обновляется каждые 5 лет; заявку на второй монитор нужно подавать руководителю [Д05#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q05", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.55, "prompt_tokens": 317, "completion_tokens": 38, "answer": " Ноутбук обновляется каждые 3 года с даты выдачи [Д05#1]. Монитор обновляется каждые 5 лет; заявку на второй монитор нужно подавать руководителю [Д05#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q05", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.18, "prompt_tokens": 98, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q05", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.11, "prompt_tokens": 98, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q06", "condition": "A", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 1.14, "prompt_tokens": 581, "completion_tokens": 54, "answer": " Время реакции и решения для заявок:\n* Critical: реакция — 30 минут, решение — 4 часа [Д06#2].\n* Major: реакция — 2 часа, решение — 1 рабочий день [Д06#2]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q06", "condition": "A", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.79, "prompt_tokens": 581, "completion_tokens": 54, "answer": " Время реакции и решения для заявок:\n* Critical: реакция — 30 минут, решение — 4 часа [Д06#2].\n* Major: реакция — 2 часа, решение — 1 рабочий день [Д06#2]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q06", "condition": "B", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.18, "prompt_tokens": 98, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q06", "condition": "B", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.12, "prompt_tokens": 98, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q07", "condition": "A", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 1.11, "prompt_tokens": 651, "completion_tokens": 44, "answer": " Тариф «Вектор-Офис Standard» стоит 2400 рублей в месяц [Д08#1]. При годовой оплате предоставляется скидка 10% от тарифа [Д08#2]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q07", "condition": "A", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 0.64, "prompt_tokens": 651, "completion_tokens": 44, "answer": " Тариф «Вектор-Офис Standard» стоит 2400 рублей в месяц [Д08#1]. При годовой оплате предоставляется скидка 10% от тарифа [Д08#2]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q07", "condition": "B", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 0.19, "prompt_tokens": 106, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q07", "condition": "B", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 0.12, "prompt_tokens": 106, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q08", "condition": "A", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 1.1, "prompt_tokens": 558, "completion_tokens": 47, "answer": " Ежегодный отпуск составляет 28 календарных дней [Д09#1]. Заявление на отпуск подаётся в СЭД «Дирекум» не позднее чем за 2 недели до начала отпуска [Д09#3]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q08", "condition": "A", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 0.68, "prompt_tokens": 558, "completion_tokens": 47, "answer": " Ежегодный отпуск составляет 28 календарных дней [Д09#1]. Заявление на отпуск подаётся в СЭД «Дирекум» не позднее чем за 2 недели до начала отпуска [Д09#3]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q08", "condition": "B", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 0.18, "prompt_tokens": 99, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q08", "condition": "B", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 0.11, "prompt_tokens": 99, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q09", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.99, "prompt_tokens": 577, "completion_tokens": 39, "answer": " Чтобы получить второй монитор, необходимо подать заявку через сервис HelpDesk в категории «Оборудование» [Д05#2]. Второй монитор выдаётся по заявке руководителя [Д05#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q09", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.57, "prompt_tokens": 577, "completion_tokens": 39, "answer": " Чтобы получить второй монитор, необходимо подать заявку через сервис HelpDesk в категории «Оборудование» [Д05#2]. Второй монитор выдаётся по заявке руководителя [Д05#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q09", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.71, "prompt_tokens": 104, "completion_tokens": 45, "answer": " Для получения второго монитора удалённому сотруднику необходимо обратиться в отдел закупок или к своему непосредственному руководителю. Учитывайте внутренние регламенты компании «Вектор-Системы» по выдаче оборудования и утверждённые бюджеты на технику."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q09", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.64, "prompt_tokens": 104, "completion_tokens": 45, "answer": " Для получения второго монитора удалённому сотруднику необходимо обратиться в отдел закупок или к своему непосредственному руководителю. Учитывайте внутренние регламенты компании «Вектор-Системы» по выдаче оборудования и утверждённые бюджеты на технику."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q10", "condition": "A", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 1.42, "prompt_tokens": 631, "completion_tokens": 65, "answer": " Заявка на командировку оформляется в СЭД «Дирекум» не позднее чем за 5 рабочих дней до выезда [Д03#2].\nАвансовый отчёт сдаётся в течение 3 рабочих дней после возвращения [Д03#3]. [Данных по лимиту расходов на гостиницу в базе знаний нет]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q10", "condition": "A", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 0.95, "prompt_tokens": 631, "completion_tokens": 65, "answer": " Заявка на командировку оформляется в СЭД «Дирекум» не позднее чем за 5 рабочих дней до выезда [Д03#2].\nАвансовый отчёт сдаётся в течение 3 рабочих дней после возвращения [Д03#3]. [Данных по лимиту расходов на гостиницу в базе знаний нет]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q10", "condition": "B", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 0.21, "prompt_tokens": 116, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q10", "condition": "B", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 0.12, "prompt_tokens": 116, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q11", "condition": "A", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 1.25, "prompt_tokens": 686, "completion_tokens": 53, "answer": " Цикл согласования договора — не более 5 рабочих дней [Д10#2].\nЦена тарифа «Вектор-Офис Enterprise» — 4900 рублей в месяц [Д08#1]. Данных о скидках в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q11", "condition": "A", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 0.77, "prompt_tokens": 686, "completion_tokens": 53, "answer": " Цикл согласования договора — не более 5 рабочих дней [Д10#2].\nЦена тарифа «Вектор-Офис Enterprise» — 4900 рублей в месяц [Д08#1]. Данных о скидках в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q11", "condition": "B", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 0.19, "prompt_tokens": 110, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q11", "condition": "B", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 0.12, "prompt_tokens": 110, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q12", "condition": "A", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.55, "prompt_tokens": 504, "completion_tokens": 11, "answer": " Данных по этому вопросу в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q12", "condition": "A", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.17, "prompt_tokens": 504, "completion_tokens": 11, "answer": " Данных по этому вопросу в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q12", "condition": "B", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.16, "prompt_tokens": 93, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q12", "condition": "B", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.12, "prompt_tokens": 93, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q13", "condition": "A", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.22, "prompt_tokens": 153, "completion_tokens": 11, "answer": " Данных по этому вопросу в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q13", "condition": "A", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.16, "prompt_tokens": 153, "completion_tokens": 11, "answer": " Данных по этому вопросу в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q13", "condition": "B", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.15, "prompt_tokens": 95, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q13", "condition": "B", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.12, "prompt_tokens": 95, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q14", "condition": "A", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.56, "prompt_tokens": 625, "completion_tokens": 11, "answer": " Данных по этому вопросу в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q14", "condition": "A", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.16, "prompt_tokens": 625, "completion_tokens": 11, "answer": " Данных по этому вопросу в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q14", "condition": "B", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.16, "prompt_tokens": 97, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q14", "condition": "B", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.12, "prompt_tokens": 97, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q15", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.42, "prompt_tokens": 514, "completion_tokens": 11, "answer": " Данных по этому вопросу в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q15", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.16, "prompt_tokens": 514, "completion_tokens": 11, "answer": " Данных по этому вопросу в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q15", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.16, "prompt_tokens": 93, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q15", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.12, "prompt_tokens": 93, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q16", "condition": "A", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.4, "prompt_tokens": 364, "completion_tokens": 11, "answer": " Данных по этому вопросу в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q16", "condition": "A", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.16, "prompt_tokens": 364, "completion_tokens": 11, "answer": " Данных по этому вопросу в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q16", "condition": "B", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.16, "prompt_tokens": 96, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q16", "condition": "B", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.12, "prompt_tokens": 96, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q17", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 0.68, "prompt_tokens": 597, "completion_tokens": 20, "answer": " Нет, неверно. Пароль нужно менять каждые 90 дней [Д02#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q17", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 0.29, "prompt_tokens": 597, "completion_tokens": 20, "answer": " Нет, неверно. Пароль нужно менять каждые 90 дней [Д02#1]."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q17", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 0.18, "prompt_tokens": 100, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q17", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 0.12, "prompt_tokens": 100, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q18", "condition": "A", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.57, "prompt_tokens": 549, "completion_tokens": 11, "answer": " Данных по этому вопросу в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q18", "condition": "A", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.16, "prompt_tokens": 549, "completion_tokens": 11, "answer": " Данных по этому вопросу в базе знаний нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q18", "condition": "B", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.18, "prompt_tokens": 98, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "YandexGPT-5-Lite-8B-instruct-Q4_K_M", "question": "Q18", "condition": "B", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.11, "prompt_tokens": 98, "completion_tokens": 8, "answer": " Данных по этому вопросу нет."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q01", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 1.75, "prompt_tokens": 1745, "completion_tokens": 25, "answer": "Испытательный срок обычного сотрудника — 3 месяца, для руководителей подразделений — 6 месяцев. [Д01]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q01", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 0.3, "prompt_tokens": 1745, "completion_tokens": 25, "answer": "Испытательный срок обычного сотрудника — 3 месяца, для руководителей подразделений — 6 месяцев. [Д01]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q01", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 0.47, "prompt_tokens": 1191, "completion_tokens": 33, "answer": "Данных по этому вопросу нет. Испытательный срок для обычного сотрудника и руководителя регулируется внутренними регламентами компании «Вектор-Системы», которые могут отличаться."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q01", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 0.37, "prompt_tokens": 1191, "completion_tokens": 33, "answer": "Данных по этому вопросу нет. Испытательный срок для обычного сотрудника и руководителя регулируется внутренними регламентами компании «Вектор-Системы», которые могут отличаться."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q02", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.57, "prompt_tokens": 1473, "completion_tokens": 28, "answer": "Минимальная длина пароля — 12 знаков, ротация — каждые 90 дней. [Д02#1]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q02", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.29, "prompt_tokens": 1473, "completion_tokens": 26, "answer": "Пароль нужно менять каждые 90 дней, минимальная длина — 12 знаков. [Д02#1]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q02", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.51, "prompt_tokens": 1190, "completion_tokens": 37, "answer": "Данных по этому вопросу нет. Обычно в «Вектор-Системах» пароль меняют раз в 3–6 месяцев, минимальная длина — 8–12 символов."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q02", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.4, "prompt_tokens": 1190, "completion_tokens": 37, "answer": "Данных по этому вопросу нет. Обычно в «Вектор-Системах» пароль меняют раз в 3–6 месяцев, минимальная длина — 8–12 символов."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q03", "condition": "A", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 1.22, "prompt_tokens": 1738, "completion_tokens": 70, "answer": "Суточные — 1500 рублей за каждый день поездки, включая дни в пути. [Д03#1]\nЛимит на гостиницу — не дороже 6500 рублей за ночь; для городов присутствия клиента — 8000 рублей по согласованию с руководителем. [Д03#1]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q03", "condition": "A", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 0.73, "prompt_tokens": 1738, "completion_tokens": 68, "answer": "Суточные — 1500 рублей за каждый день поездки, включая дни в пути. [Д03#1]\nГостиница — не дороже 6500 рублей за ночь; для городов присутствия клиента — лимит 8000 рублей по согласованию с руководителем. [Д03#1]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q03", "condition": "B", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 0.26, "prompt_tokens": 1192, "completion_tokens": 14, "answer": "Данных по суточным и лимиту на гостиницу нет."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q03", "condition": "B", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 0.17, "prompt_tokens": 1192, "completion_tokens": 14, "answer": "Данных по суточным и лимиту на гостиницу нет."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q04", "condition": "A", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 1.01, "prompt_tokens": 1768, "completion_tokens": 46, "answer": "Удалёнщик обязан быть в офисе не менее 2 дней в неделю (1.1). Ядро доступности — 11:00–15:00 (1.3). [Д04]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q04", "condition": "A", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 0.52, "prompt_tokens": 1768, "completion_tokens": 46, "answer": "Удалёнщик обязан быть в офисе не менее 2 дней в неделю (1.1). Ядро доступности — 11:00–15:00 (1.3). [Д04]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q04", "condition": "B", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 0.36, "prompt_tokens": 1197, "completion_tokens": 23, "answer": "Данных по этому вопросу нет. Уточните, пожалуйста, в каком документе или регламенте содержится информация."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q04", "condition": "B", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 0.65, "prompt_tokens": 1197, "completion_tokens": 62, "answer": "Данных по этому вопросу нет. Обычно в «Вектор-Системах» удалёнщики обязаны быть в офисе 2 дня в неделю, а ядро доступности — с 10:00 до 17:00. Точные условия нужно уточнять в локальных регламентах."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q05", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.86, "prompt_tokens": 1436, "completion_tokens": 61, "answer": "Ноутбук обновляется каждые 3 года с даты выдачи [Д05#1]. Монитор — каждые 5 лет; второй монитор выдаётся по заявке руководителя [Д05#1]. Заявки на периферию подаются при выходе из строя [Д05#1]."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q05", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.61, "prompt_tokens": 1436, "completion_tokens": 61, "answer": "Ноутбук обновляется каждые 3 года с даты выдачи [Д05#1]. Монитор — каждые 5 лет; второй монитор выдаётся по заявке руководителя [Д05#1]. Заявки на периферию подаются при выходе из строя [Д05#1]."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q05", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.52, "prompt_tokens": 1192, "completion_tokens": 40, "answer": "Данных по этому вопросу нет. Заявки на обновление техники обычно подаются через внутренний портал или HR-специалисту. Точные сроки и регламенты утверждаются внутренними документами компании."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q05", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.39, "prompt_tokens": 1192, "completion_tokens": 37, "answer": "Данных по этому вопросу нет. Заявки на обновление техники обычно подаются через внутренний портал или HR-специалисту. График обновления зависит от политики компании и состояния оборудования."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q06", "condition": "A", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.62, "prompt_tokens": 1707, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д06#2]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q06", "condition": "A", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.22, "prompt_tokens": 1707, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д06#2]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q06", "condition": "B", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.2, "prompt_tokens": 1192, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q06", "condition": "B", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.62, "prompt_tokens": 1192, "completion_tokens": 56, "answer": "Данных по этому вопросу нет. Обычно в ИТ-компаниях время реакции на Critical — до 15 минут, решения — до 1 часа, на Major — до 4 часов и до 1 рабочего дня соответственно. Точные сроки утверждаются внутренними регламентами."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q07", "condition": "A", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 1.06, "prompt_tokens": 1784, "completion_tokens": 48, "answer": "Тариф «Вектор-Офис Standard» стоит 2400 рублей в месяц. При годовой оплате предоставляется скидка 10 % от тарифа. [Д08#1]\n[Д08#2]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q07", "condition": "A", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 0.56, "prompt_tokens": 1784, "completion_tokens": 48, "answer": "Тариф «Вектор-Офис Standard» стоит 2400 рублей в месяц. При годовой оплате предоставляется скидка 10 % от тарифа. [Д08#1]\n[Д08#2]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q07", "condition": "B", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 0.37, "prompt_tokens": 1200, "completion_tokens": 24, "answer": "Данных по стоимости тарифа «Вектор-Офис Standard» и условиям скидок за годовую оплату нет."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q07", "condition": "B", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 0.28, "prompt_tokens": 1200, "completion_tokens": 24, "answer": "Данных по стоимости тарифа «Вектор-Офис Standard» и условиям скидок за годовую оплату нет."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q08", "condition": "A", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 1.0, "prompt_tokens": 1682, "completion_tokens": 50, "answer": "Ежегодный отпуск — 28 календарных дней. Заявление подаётся в СЭД «Дирекум» не позднее чем за 2 недели до начала отпуска. [Д09#1], [Д09#3]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q08", "condition": "A", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 0.54, "prompt_tokens": 1682, "completion_tokens": 50, "answer": "Ежегодный отпуск — 28 календарных дней. Заявление подаётся в СЭД «Дирекум» не позднее чем за 2 недели до начала отпуска. [Д09#1], [Д09#3]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q08", "condition": "B", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 0.19, "prompt_tokens": 1192, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q08", "condition": "B", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 0.59, "prompt_tokens": 1192, "completion_tokens": 54, "answer": "Данных по этому вопросу нет. Обычно в компании «Вектор-Системы» ежегодный отпуск составляет 28 календарных дней, а заявление подаётся за 14–15 календарных дней до начала. Точные условия уточняются во внутренних регламентах."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q09", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.63, "prompt_tokens": 1716, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д05#1]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q09", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.22, "prompt_tokens": 1716, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д05#1]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q09", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.71, "prompt_tokens": 1197, "completion_tokens": 55, "answer": "Удалёнщик подаёт заявку через корпоративный портал на имя руководителя отдела. В заявке указывает модель, серийный номер и обоснование (работа с клиентскими проектами). Учитывайте лимиты бюджета, согласование с IT-отделом и правила учёта оборудования."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q09", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.55, "prompt_tokens": 1197, "completion_tokens": 52, "answer": "Удалёнщик подаёт заявку через корпоративный портал на имя руководителя отдела. В заявке указывает модель, серийный номер и назначение (клиентский проект). Учитывайте лимиты бюджета, согласование с IT-отделом и правила эксплуатации оборудования."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q10", "condition": "A", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 1.31, "prompt_tokens": 1761, "completion_tokens": 81, "answer": "Заявку в СЭД «Дирекум» подают не позднее чем за 5 рабочих дней до выезда [Д03#2]. К отчёту прилагаются чеки; чеки без реквизитов отеля не принимаются [Д03#3]. Перерасход более 10 % от лимита возмещается только с письменной визы руководителя [Д03#3]."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q10", "condition": "A", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 0.85, "prompt_tokens": 1761, "completion_tokens": 81, "answer": "Заявку в СЭД «Дирекум» подают не позднее чем за 5 рабочих дней до выезда [Д03#2]. К отчёту прилагаются чеки; чеки без реквизитов отеля не принимаются [Д03#3]. Перерасход более 10 % от лимита возмещается только с письменной визы руководителя [Д03#3]."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q10", "condition": "B", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 0.73, "prompt_tokens": 1213, "completion_tokens": 57, "answer": "Данных по этому вопросу нет. Обычно заявку в СЭД «Дирекум» подают за 3–5 рабочих дней до поездки. Лимит на гостиницу устанавливается внутренними регламентами, авансовый отчёт сдаётся в течение 3 рабочих дней после возвращения."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q10", "condition": "B", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 0.6, "prompt_tokens": 1213, "completion_tokens": 61, "answer": "Данных по этому вопросу нет. Обычно заявку в СЭД «Дирекум» подают за 3–5 рабочих дней до командировки. Лимит на гостиницу определяется внутренними регламентами, авансовый отчёт с подтверждающими документами — в течение 3 рабочих дней после возвращения."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q11", "condition": "A", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 0.7, "prompt_tokens": 1820, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д10#2]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q11", "condition": "A", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 0.21, "prompt_tokens": 1820, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д10#2]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q11", "condition": "B", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 0.6, "prompt_tokens": 1206, "completion_tokens": 43, "answer": "Данных по срокам цикла согласования нет. Цена тарифа Enterprise с годовой оплатой — 1 200 000 ₽, при единовременной оплате предоставляется скидка 15%."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q11", "condition": "B", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 0.45, "prompt_tokens": 1206, "completion_tokens": 43, "answer": "Данных по срокам цикла согласования нет. Цена тарифа Enterprise с годовой оплатой — 1 200 000 ₽. При единовременной оплате предоставляется скидка 15%."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q12", "condition": "A", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.54, "prompt_tokens": 1628, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д01#0]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q12", "condition": "A", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.2, "prompt_tokens": 1628, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д01#0]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q12", "condition": "B", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.41, "prompt_tokens": 1187, "completion_tokens": 29, "answer": "Данных по этому вопросу нет. Размер годовой премии сотрудников определяется внутренними регламентами компании и зависит от результатов работы, должности и других факторов."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q12", "condition": "B", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.31, "prompt_tokens": 1187, "completion_tokens": 29, "answer": "Данных по этому вопросу нет. Размер годовой премии сотрудников определяется внутренними регламентами компании и зависит от должности, результатов работы и других факторов."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q13", "condition": "A", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.36, "prompt_tokens": 1253, "completion_tokens": 21, "answer": "Данных по этому вопросу в базе знаний нет. [Д01] [Д02]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q13", "condition": "A", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.19, "prompt_tokens": 1253, "completion_tokens": 16, "answer": "Данных по этому вопросу в базе знаний нет. [Д01]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q13", "condition": "B", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.6, "prompt_tokens": 1189, "completion_tokens": 48, "answer": "Данных по этому вопросу нет. Обычно оформление корпоративной SIM-карты происходит через корпоративный отдел или IT-службу компании с предоставлением паспорта и заявления. Конкретные процедуры зависят от внутренних регламентов «Вектор-Систем»."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q13", "condition": "B", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.35, "prompt_tokens": 1189, "completion_tokens": 34, "answer": "Данных по этому вопросу нет. Обычно оформление корпоративной SIM-карты происходит через отдел кадров или IT-департамент компании с предоставлением паспорта сотрудника и заявления."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q14", "condition": "A", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.62, "prompt_tokens": 1759, "completion_tokens": 16, "answer": "Данных по этому вопросу в базе знаний нет. [Д08]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q14", "condition": "A", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.19, "prompt_tokens": 1759, "completion_tokens": 16, "answer": "Данных по этому вопросу в базе знаний нет. [Д08]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q14", "condition": "B", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 1192, "completion_tokens": 32, "answer": "Данных по стоимости тарифа «Вектор-Офис Premium» нет. Уточнить цену можно у отдела продаж компании «Вектор-Системы»."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q14", "condition": "B", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.35, "prompt_tokens": 1192, "completion_tokens": 32, "answer": "Данных по стоимости тарифа «Вектор-Офис Premium» нет. Уточнить цену можно у отдела продаж компании «Вектор-Системы»."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q15", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.55, "prompt_tokens": 1630, "completion_tokens": 16, "answer": "Данных по этому вопросу в базе знаний нет. [Д01]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q15", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.19, "prompt_tokens": 1630, "completion_tokens": 16, "answer": "Данных по этому вопросу в базе знаний нет. [Д01]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q15", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.37, "prompt_tokens": 1188, "completion_tokens": 25, "answer": "Данных по этому вопросу нет. Оформление материнского капитала регулируется законодательством РФ и осуществляется через Социальный фонд России."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q15", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.28, "prompt_tokens": 1188, "completion_tokens": 25, "answer": "Данных по этому вопросу нет. Оформление материнского капитала регулируется законодательством РФ и осуществляется через Социальный фонд России."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q16", "condition": "A", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 1478, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д10#1]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q16", "condition": "A", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.21, "prompt_tokens": 1478, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д10#1]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q16", "condition": "B", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.19, "prompt_tokens": 1191, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q16", "condition": "B", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.1, "prompt_tokens": 1191, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q17", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 0.69, "prompt_tokens": 1735, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д02#1]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q17", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 0.22, "prompt_tokens": 1735, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д02#1]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q17", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 0.52, "prompt_tokens": 1194, "completion_tokens": 38, "answer": "Данных по этому вопросу нет. В «Вектор-Системах» действуют внутренние регламенты информационной безопасности, но частота смены паролей определяется ими, а не общим правилом."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q17", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 0.42, "prompt_tokens": 1194, "completion_tokens": 39, "answer": "Данных по этому вопросу нет. В «Вектор-Системах» действуют внутренние регламенты информационной безопасности, но частота смены паролей определяется ими, а не универсальным правилом."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q18", "condition": "A", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.63, "prompt_tokens": 1672, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д09#1]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q18", "condition": "A", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.22, "prompt_tokens": 1672, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет. [Д09#1]"}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q18", "condition": "B", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.6, "prompt_tokens": 1193, "completion_tokens": 45, "answer": "Данных по этому вопросу нет. Согласно ТК РФ, ежегодный оплачиваемый отпуск можно делить на части, но с согласия работника и работодателя, при этом хотя бы одна часть должна быть не менее 14 дней."}
{"model": "M6-GigaChat3.1-10B-A1.8B-q8_0", "question": "Q18", "condition": "B", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.52, "prompt_tokens": 1193, "completion_tokens": 50, "answer": "Данных по этому вопросу нет. Согласно Трудовому кодексу РФ, ежегодный оплачиваемый отпуск можно делить на части, но с согласия работника и работодателя. При этом хотя бы одна из частей должна быть не менее 14 календарных дней."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q01", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 3.74, "prompt_tokens": 757, "completion_tokens": 251, "answer": "Обычный сотрудник – 3 месяца; руководитель подразделения – 6 месяцев. [Д01#1]"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q01", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 1.67, "prompt_tokens": 757, "completion_tokens": 147, "answer": "Обычным сотрудникам испытательный срок составляет 3 месяца, а для руководителей подразделений — 6 месяцев. (Д01#1)"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q01", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 1.09, "prompt_tokens": 169, "completion_tokens": 82, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q01", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 2.81, "prompt_tokens": 169, "completion_tokens": 252, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q02", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 3.93, "prompt_tokens": 458, "completion_tokens": 324, "answer": "Пароль менять каждые 90 дней ( [Д02#1] ); минимальная длина 12 знаков ( [Д02#1] )."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q02", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 1.77, "prompt_tokens": 458, "completion_tokens": 158, "answer": "Пароль необходимо менять каждые 90 дней, а минимальная длина — 12 знаков. [Д02#1]"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q02", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 1.64, "prompt_tokens": 165, "completion_tokens": 131, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q02", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 1.16, "prompt_tokens": 165, "completion_tokens": 103, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q03", "condition": "A", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 4.24, "prompt_tokens": 734, "completion_tokens": 333, "answer": "Суточные: 1500 руб. за каждый день поездки, включая дни в пути. Гостиница: не дороже 6500 руб. за ночь; в городах присутствия клиента лимит 8000 руб. при согласовании с руководителем. [Д03]"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q03", "condition": "A", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 4.53, "prompt_tokens": 734, "completion_tokens": 400, "answer": "Суточные 1500 рублей за каждый день поездки, включая дни в пути (Д03). Лимит гостиницы — не дороже 6500"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q03", "condition": "B", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 2.35, "prompt_tokens": 168, "completion_tokens": 194, "answer": "Данных по этому вопросу нет"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q03", "condition": "B", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 2.45, "prompt_tokens": 168, "completion_tokens": 218, "answer": "Данных по этому вопросу нет"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q04", "condition": "A", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 4.41, "prompt_tokens": 750, "completion_tokens": 347, "answer": "Удалёнщик обязан быть в офисе не менее **2 дней в неделю** [Д04#1].  \nЯдро доступности составляет **11:00–15:00** [Д04#1]."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q04", "condition": "A", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 3.57, "prompt_tokens": 750, "completion_tokens": 316, "answer": "Удалёнщик должен присутствовать в офисе не менее 2 дней в неделю. Ядро доступности — 11:00–15:00, во время которого он обязан быть доступен по видео и телефону. [Д04#1]"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q04", "condition": "B", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 1.83, "prompt_tokens": 174, "completion_tokens": 148, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q04", "condition": "B", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 1.07, "prompt_tokens": 174, "completion_tokens": 95, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q05", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 4.76, "prompt_tokens": 429, "completion_tokens": 400, "answer": ""}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q05", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 4.48, "prompt_tokens": 429, "completion_tokens": 400, "answer": "Рабочий ноутбук обновляется каждые 3 года ("}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q05", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 1.79, "prompt_tokens": 167, "completion_tokens": 144, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q05", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 1.9, "prompt_tokens": 167, "completion_tokens": 170, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q06", "condition": "A", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 3.67, "prompt_tokens": 686, "completion_tokens": 284, "answer": "Critical — реакция 30 минут, решение 4 часа [Д06#2].  \nMajor — реакция 2 часа, решение 1 рабочий день [Д06#2]."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q06", "condition": "A", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 2.88, "prompt_tokens": 686, "completion_tokens": 255, "answer": "Critical: реакция 30 минут, решение 4 часа [Д06#2].  \nMajor: реакция 2 часа, решение 1 рабочий день [Д06#2]."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q06", "condition": "B", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 2.12, "prompt_tokens": 166, "completion_tokens": 173, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q06", "condition": "B", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 1.72, "prompt_tokens": 166, "completion_tokens": 153, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q07", "condition": "A", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 5.0, "prompt_tokens": 751, "completion_tokens": 400, "answer": "Тариф «Вектор‑Офис Standard» — 2400 руб. за пользователя в месяц ( [Д08#1] ). При годовой оплате действует скидка "}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q07", "condition": "A", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 1.41, "prompt_tokens": 751, "completion_tokens": 124, "answer": "Тариф «Вектор‑Офис Standard» стоит 2400 рублей в месяц (пользователя) [Д08#1]. При годовой оплате действует скидка 10 % от тарифа [Д08#2]."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q07", "condition": "B", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 2.04, "prompt_tokens": 177, "completion_tokens": 166, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q07", "condition": "B", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 0.98, "prompt_tokens": 177, "completion_tokens": 87, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q08", "condition": "A", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 3.35, "prompt_tokens": 669, "completion_tokens": 257, "answer": "Ежегодный отпуск составляет 28 календарных дней. Заявление подаётся в СЭД «Дирекум» не позднее чем за 2 недели до начала отпуска. [Д09#1] [Д09#3]"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q08", "condition": "A", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 4.51, "prompt_tokens": 669, "completion_tokens": 400, "answer": ""}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q08", "condition": "B", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 1.84, "prompt_tokens": 168, "completion_tokens": 149, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q08", "condition": "B", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 0.78, "prompt_tokens": 168, "completion_tokens": 70, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q09", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 4.98, "prompt_tokens": 705, "completion_tokens": 400, "answer": ""}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q09", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 4.52, "prompt_tokens": 705, "completion_tokens": 400, "answer": "Для запроса второго монитора через HelpDesk — «Оборудование» (с согласованием руководителя).  \nСрок исполнения – 5 рабочих дней (1 день при критичном проекте).  \nКлиентские данные можно работать только на выдаваемом оборудовании, личный монитор запрещён.  \n\n[Д05#1] [Д05#2] [Д"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q09", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 4.65, "prompt_tokens": 172, "completion_tokens": 400, "answer": ""}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q09", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 4.48, "prompt_tokens": 172, "completion_tokens": 400, "answer": ""}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q10", "condition": "A", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 5.01, "prompt_tokens": 756, "completion_tokens": 400, "answer": ""}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q10", "condition": "A", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 4.52, "prompt_tokens": 756, "completion_tokens": 400, "answer": ""}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q10", "condition": "B", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 2.78, "prompt_tokens": 193, "completion_tokens": 232, "answer": "Данных по этому вопросу нет"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q10", "condition": "B", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 1.77, "prompt_tokens": 193, "completion_tokens": 157, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q11", "condition": "A", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 5.02, "prompt_tokens": 807, "completion_tokens": 400, "answer": ""}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q11", "condition": "A", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 4.52, "prompt_tokens": 807, "completion_tokens": 400, "answer": ""}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q11", "condition": "B", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 1.25, "prompt_tokens": 184, "completion_tokens": 95, "answer": "Данных по этому вопросу нет"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q11", "condition": "B", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 1.0, "prompt_tokens": 184, "completion_tokens": 89, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q12", "condition": "A", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 2.48, "prompt_tokens": 574, "completion_tokens": 192, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q12", "condition": "A", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 4.49, "prompt_tokens": 574, "completion_tokens": 400, "answer": ""}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q12", "condition": "B", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 1.0, "prompt_tokens": 162, "completion_tokens": 74, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q12", "condition": "B", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.75, "prompt_tokens": 162, "completion_tokens": 67, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q13", "condition": "A", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 4.66, "prompt_tokens": 229, "completion_tokens": 400, "answer": ""}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q13", "condition": "A", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 2.08, "prompt_tokens": 229, "completion_tokens": 185, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q13", "condition": "B", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 3.48, "prompt_tokens": 161, "completion_tokens": 297, "answer": "Для оформления корпоративной SIM‑карты:\n\n1. Зайти в портал IT‑службы и заполнить форму «Заказ корпоративной SIM».  \n2. Выбрать оператор (Vodafone, MTS, Beeline и т.д.) и указать назначение – рабочий телефон.  \n3. Прикрепить электронный адрес компании и отправить заявку.  \n4. После одобрения реквизиты оформляются в системе закупок, сотрудник получает SIM, PIN‑код и инструкцию по активации.  \n5. Зарегистрировать SIM в CMDB и привязать её к профилю сотрудника."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q13", "condition": "B", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 3.36, "prompt_tokens": 161, "completion_tokens": 300, "answer": "Для оформления корпоративной SIM‑карты сотрудник открывает заявку в системе ИТ‑поддержки, указывает нужного оператора и номер телефона, получает одобрение руководителя. После утверждения ИТ‑отдел выдаёт SIM‑карту и переводит её в активный статус."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q14", "condition": "A", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 4.72, "prompt_tokens": 716, "completion_tokens": 376, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q14", "condition": "A", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 2.23, "prompt_tokens": 716, "completion_tokens": 197, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q14", "condition": "B", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 1.98, "prompt_tokens": 167, "completion_tokens": 161, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q14", "condition": "B", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.95, "prompt_tokens": 167, "completion_tokens": 84, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q15", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 2.02, "prompt_tokens": 609, "completion_tokens": 143, "answer": "Данных по этому вопросу в базе знаний нет"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q15", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 2.53, "prompt_tokens": 609, "completion_tokens": 225, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q15", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 3.37, "prompt_tokens": 163, "completion_tokens": 286, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q15", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 1.79, "prompt_tokens": 163, "completion_tokens": 159, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q16", "condition": "A", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 2.8, "prompt_tokens": 454, "completion_tokens": 224, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q16", "condition": "A", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 2.28, "prompt_tokens": 454, "completion_tokens": 202, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q16", "condition": "B", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 1.5, "prompt_tokens": 165, "completion_tokens": 118, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q16", "condition": "B", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 1.38, "prompt_tokens": 165, "completion_tokens": 123, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q17", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 2.29, "prompt_tokens": 724, "completion_tokens": 160, "answer": "Нет, пароль меняется каждые 90 дней, а не 60 дней (Д02)."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q17", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 1.97, "prompt_tokens": 724, "completion_tokens": 174, "answer": "Нет, в базе указана ротация пароля каждые 90 дней, а не 60  дней. [Д02#1]"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q17", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 1.99, "prompt_tokens": 168, "completion_tokens": 162, "answer": "Данных по этому вопросу нет"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q17", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 1.04, "prompt_tokens": 168, "completion_tokens": 92, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q18", "condition": "A", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 4.95, "prompt_tokens": 642, "completion_tokens": 400, "answer": ""}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q18", "condition": "A", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 2.63, "prompt_tokens": 642, "completion_tokens": 232, "answer": "Данных по этому вопросу в базе знаний нет. [Д09]"}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q18", "condition": "B", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 1.73, "prompt_tokens": 166, "completion_tokens": 139, "answer": "Данных по этому вопросу нет."}
{"model": "M7-gpt-oss-20b-Q8_0", "question": "Q18", "condition": "B", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 1.73, "prompt_tokens": 166, "completion_tokens": 154, "answer": "Данных по этому вопросу нет."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q01", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 2.55, "prompt_tokens": 733, "completion_tokens": 48, "answer": "Испытательный срок для обычного сотрудника ООО «Вектор-Системы» составляет 3 месяца [Д01#1], для руководителей подразделений — 6 месяцев [Д01#1]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q01", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 1.0, "prompt_tokens": 733, "completion_tokens": 32, "answer": "Испытательный срок для обычного сотрудника составляет 3 месяца, для руководителей подразделений — 6 месяцев [Д01#1]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q01", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 2.05, "prompt_tokens": 115, "completion_tokens": 64, "answer": "В «Вектор-Системах» для всех новых сотрудников предусмотрен испытательный срок:\n- 3 месяца для обычных сотрудников;\n- 6 месяцев для руководителей, занимающих позиции руководителя отдела и выше.\n\nПосле успешного прохождения этого периода сотрудник становится штатным сотрудником компании."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q01", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 3.08, "prompt_tokens": 115, "completion_tokens": 111, "answer": "Данные по этому вопросу уточняются согласно внутренним положениям и договорам с сотрудниками.\n\nОбычное правило для обычных сотрудников: испытательный срок составляет 3 месяца.\n\nДля руководящих позиций:\n\n- Руководитель отдела: 6 месяцев\n- Руководитель отдела, подчиняющийся директору: 9 месяцев\n- Заместитель директора: 12 месяцев\n- Директор: 1 год\n\nПри этом все сроки могут пересматриваться при необходимости и согласовании с HR."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q02", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 2.62, "prompt_tokens": 423, "completion_tokens": 70, "answer": "Минимальная длина пароля должна составлять 12 знаков, а в нём обязательно должны присутствовать цифры и спецсимволы [Д02#1.1]. Ротация паролей происходит каждые 90 дней [Д02#1.2].\n\nЕсли есть другие вопросы, пожалуйста, задавайте."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q02", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 1.59, "prompt_tokens": 423, "completion_tokens": 55, "answer": "Пароль необходимо менять каждые 90 дней [Д02#1]. Минимальная длина пароля — 12 символов, при этом он должен содержать как минимум одну цифру и один специальный символ [Д02#1]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q02", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 2.62, "prompt_tokens": 112, "completion_tokens": 85, "answer": "Согласно внутренним регламентам «Вектора-Системы», пароль сотрудников следует менять каждые полгода. Минимальная длина пароля составляет 8 символов, включая хотя бы одну цифру, одну строчную букву и один специальный символ, например, !, @, #. Рекомендуется также использование не менее 10 символов и регулярное изменение пароля для повышения безопасности."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q02", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 2.42, "prompt_tokens": 112, "completion_tokens": 87, "answer": "По внутреннему регламенту, устанавливаемому отделом информационной безопасности, пароли пользователей компании «Вектор-Системы» необходимо менять каждые 90 дней. Минимальная длина пароля составляет 12 символов, включающих латинские буквы в верхнем и нижнем регистре, цифры и специальные символы. Это требование направлено на повышение уровня безопасности корпоративной информационной системы."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q03", "condition": "A", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 3.45, "prompt_tokens": 717, "completion_tokens": 81, "answer": "Суточные при командировке составляют 1500 рублей за каждый день, включая дни в пути. Для городов без присутствия клиента лимит по гостинице — не более 6500 рублей за ночь. В городах, где есть клиенты компании, лимит по гостинице может быть увеличен до 8000 рублей с согласованием руководителя [Д03#1]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q03", "condition": "A", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 2.58, "prompt_tokens": 717, "completion_tokens": 89, "answer": "Суточные при командировке в ООО «Вектор-Системы» составляют 1500 рублей за день, включая дни в пути [Д03#1.1]. Гостиница не должна стоить дороже 6500 рублей за ночь; для городов присутствия клиента лимит повышается до 8000 рублей по согласованию с руководителем [Д03#1.2]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q03", "condition": "B", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 2.94, "prompt_tokens": 113, "completion_tokens": 96, "answer": "В рамках нашей политики командировочных расходов, суточные при командировках составляют 2 500 рублей в день. Однако, эти суммы могут изменяться в зависимости от региона или типа назначения.\n\nЧто касается лимита на гостиницу, он определяется исходя из уровня должности сотрудника, но не превышает 15 000 рублей за одну ночь. Также необходимо предоставлять подтверждение бронирования или оплаты гостиницы для возмещения расходов."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q03", "condition": "B", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 3.53, "prompt_tokens": 113, "completion_tokens": 128, "answer": "При командировках в компании «Вектор-Системы» действуют следующие регламенты по суточным:\n\n• Для городов России: 6000 рублей в сутки\n• Для городов СНГ: 8000 рублей в сутки\n• Для зарубежных стран: 10000 рублей (без учета региона)\n\nЛимит на гостиницу установлен в пределах:\n• 3000 рублей в сутки для городов России и СНГ\n• 5000 рублей в сутки для зарубежных стран\n\nЭти суммы могут корректироваться внутренними распоряжениями руководства."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q04", "condition": "A", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 3.5, "prompt_tokens": 738, "completion_tokens": 84, "answer": "Согласно регламенту удалённой работы ООО «Вектор-Системы», удалёнщик должен находиться в офисе не менее 2 рабочих дней в неделю (дни определяет руководитель) [Д04#1] и обязан быть доступным по видео и телефону с 11:00 до 15:00 – это и есть ядро доступности [Д04#1]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q04", "condition": "A", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 2.58, "prompt_tokens": 738, "completion_tokens": 89, "answer": "По данным регламента удалённой работы (Д04#1), удалённые сотрудники компании ООО «Вектор-Системы» обязаны находиться в офисе не менее 2 рабочих дней в неделю (определение конкретных дней осуществляется руководителем). Ядро доступности (время, в течение которого сотрудник должен быть доступен по видео и телефону) составляет 11:00–15:00."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q04", "condition": "B", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 2.78, "prompt_tokens": 120, "completion_tokens": 90, "answer": "Для сотрудников, работающих удаленно в «Вектор-Системы», действует гибкий график, согласно которому один день в неделю является очным в офисе для организации взаимодействия и укрепления командного духа. Ядро доступности устанавливается индивидуально для каждой удаленной позиции на основе специфики проекта и потребности в контакте с командой. При этом сотрудники обязаны соответствовать требованиям ядро доступности в рабочее время."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q04", "condition": "B", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 3.13, "prompt_tokens": 120, "completion_tokens": 113, "answer": "Согласно внутреннему регламенту \"Вектор-Систем\", удалённые сотрудники обязаны находиться в офисе **1 день в неделю**. В рамках этого дня предусмотрено **ядро доступности с 9:00 до 18:00** (по местному времени). В остальное время удалённая работа предполагает гибкий график при условии обеспечения выполнения установленных обязанностей и своевременного реагирования на запросы коллег.\n\nОтвет: Данных по ядру доступности удалённого сотрудника нет."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q05", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 3.63, "prompt_tokens": 386, "completion_tokens": 111, "answer": "Рабочий ноутбук обновляется каждые 3 года с даты выдачи [Д05#1.1]. Монитор — каждые 5 лет, второй монитор выдаётся по заявке руководителя [Д05#1.2]. Периферийное оборудование обновляется по заявке при выходе из строя [Д05#1.3]. Рабочие ноутбуки и гарнитуры выдаются компанией [Д04#2.1]. Заявки на второй монитор подать руководителю."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q05", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 2.61, "prompt_tokens": 386, "completion_tokens": 92, "answer": "Ноутбуки обновляются каждые 3 года с даты выдачи [Д05#1]. Мониторы — каждые 5 лет, второй монитор — по заявке руководителя [Д05#1]. Заявку на обновление или замену периферии подают по факту выхода из строя [Д05#1]. Рабочий ноутбук и гарнитуру выдает компания [Д04#2]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q05", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 3.12, "prompt_tokens": 116, "completion_tokens": 103, "answer": "Данные по графику обновления оборудования и процессу подачи заявки на замену ноутбуков и мониторов, регулируются внутренним документом «Политика обновления оборудования в «Вектор-Системах»». Обычно обновление происходит каждые 3 года для офисного ноутбука и 5 лет для настольного монитора. Заявку на замену можно подать в службу технической поддержки через внутренний портал «IT-Сервис» или лично обратиться к вашему руководителю."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q05", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 2.93, "prompt_tokens": 116, "completion_tokens": 106, "answer": "Обновление ноутбуков и мониторов в компании «Вектор-Системы» происходит каждые 3-4 года согласно установленным регламентам. Внутреннее распоряжение №6/23-03 утверждает стандартные сроки замены техники. Подать заявку на обновление можно через корпоративный портал на странице «Оборудование и техника» или передать письмо в службу IT-поддержки с указанием причины замены. Решение принимается директором IT-отдела."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q06", "condition": "A", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 3.51, "prompt_tokens": 674, "completion_tokens": 87, "answer": "Для заявок Critical (полный отказ сервиса, работа подразделения остановлена) время реакции составляет 30 минут, а решение должно быть найдено в течение 4 часов [Д06#2]. Для заявок Major (частичная работоспособность сервиса, затронуты не все пользователи) время реакции составляет 2 часа, а решение — 1 рабочий день [Д06#2]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q06", "condition": "A", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 1.65, "prompt_tokens": 674, "completion_tokens": 56, "answer": "Для Critical заявок: время реакции — 30 минут, решение — 4 часа [Д06#2.1].\nДля Major заявок: время реакции — 2 часа, решение — 1 рабочий день [Д06#2.2]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q06", "condition": "B", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 2.89, "prompt_tokens": 113, "completion_tokens": 94, "answer": "Для заявок Critical и Major в компании «Вектор-Системы» действуют следующие временные рамки:\n\n- Critical (критические):\nВремя реакции: 30 минут\nВремя решения: до 4 часов\n\n- Major (важные):\nВремя реакции: до 2 часов\nВремя решения: до 8 часов\n\nЭти параметры определены во Внутреннем Регламенте по обслуживанию и обработке заявок."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q06", "condition": "B", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 1.96, "prompt_tokens": 113, "completion_tokens": 70, "answer": "Время реакции на заявки:\n- Critical: 2 часа\n- Major: 4 часа\n\nВремя принятия решения:\n- Critical: 4 часа\n- Major: 8 часов\n\nЭти показатели установлены согласно регламенту службы поддержки. Процедура рассмотрения заявок включает этапы приоритезации, анализа и реализации решений."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q07", "condition": "A", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 3.25, "prompt_tokens": 762, "completion_tokens": 75, "answer": "Тариф «Вектор-Офис Standard» стоит 2400 рублей за пользователя в месяц [Д08#1]. При годовой оплате предоставляется скидка 10 % от тарифа [Д08#2].\n\n[Д08#0][Д08#1][Д08#2]"}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q07", "condition": "A", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 3.03, "prompt_tokens": 762, "completion_tokens": 105, "answer": "Тариф «Вектор-Офис Standard» составляет 2400 рублей за пользователя в месяц [Д08#1]. При годовой оплате предоставляется скидка 10 % [Д08#2, п. 2.1]. Дополнительные скидки предусмотрены для клиентов от 500 пользователей, но это не относится к указанному тарифу [Д08#2, п. 2.2]. [Д08#1]"}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q07", "condition": "B", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 2.57, "prompt_tokens": 125, "completion_tokens": 82, "answer": "Тариф «Вектор-Офис Standard» в настоящее время стоит 25 000 рублей в месяц. При оплате за годовый период предоставляется скидка — 12% от стоимости ежемесячного тарифа за весь годовой контракт. Таким образом, при годовой оплате общая стоимость составит 297 000 рублей."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q07", "condition": "B", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 2.21, "prompt_tokens": 125, "completion_tokens": 79, "answer": "Тариф \"Вектор-Офис Standard\" стоит 10 000 руб. в месяц. Однако для клиентов, которые выбирают ежегодную оплату, действует скидка в размере 10%. Таким образом, годовая стоимость становится равной 108 000 руб., что на 12% меньше месячной оплаты."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q08", "condition": "A", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 3.53, "prompt_tokens": 656, "completion_tokens": 92, "answer": "Ответ:\n\n28 календарных дней ежегодного отпуска предоставляется сотруднику компании ООО «Вектор-Системы» [Д09#1.1]. Заявление на отпуск подаётся в СЭД «Дирекум» не позднее, чем за 2 недели до начала отпуска [Д09#3.1].\n\n[Д09#1], [Д09#3]"}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q08", "condition": "A", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 3.75, "prompt_tokens": 656, "completion_tokens": 132, "answer": "Ежегодный оплачиваемый отпуск в ООО «Вектор-Системы» составляет 28 календарных дней [Д09#1, пункт 1.1]. Кроме того, предусмотрен дополнительный отпуск за стаж в компании: 3 дня при стаже от 5 лет и 5 дней при стаже от 10 лет [Д09#1, пункт 1.2]. Заявление на отпуск нужно подавать в СЭД «Дирекум» не позднее чем за две недели до его начала [Д09#3, пункт 3.1]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q08", "condition": "B", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 2.62, "prompt_tokens": 114, "completion_tokens": 84, "answer": "Стандартный ежегодный отпуск в «Вектор-Системах» составляет 28 календарных дней. Заявление на отпуск подается не позднее чем за 30 календарных дней до предполагаемой даты начала отпуска. Это правило может варьироваться в зависимости от трудового договора и должности сотрудника. Для детализированной информации рекомендуется обращаться в отдел кадров."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q08", "condition": "B", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 2.75, "prompt_tokens": 114, "completion_tokens": 99, "answer": "Сотрудникам компании «Вектор-Системы» полагается 28 календарных дней ежегодного оплачиваемого отпуска. Заявление на отпуск подаётся минимум за два месяца до его начала, но не позднее 45 дней. Также существует возможность подачи заявления через корпоративную систему управления персоналом «HR+» в электронном виде. Точные правила и условия могут корректироваться в зависимости от должностных обязанностей и стажа работы в компании."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q09", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 5.32, "prompt_tokens": 686, "completion_tokens": 153, "answer": "Для получения второго монитора при удалённой работе над клиентскими проектами:\n1. Подать заявку через HelpDesk, выбрав категорию «Оборудование» [Д05#2];\n2. Указать критичность проекта для ускорения рассмотрения (до 1 рабочего дня) [Д05#2];\n3. Приоритет выдаче второго монитора предоставляется сотрудникам, работающим с клиентскими проектами [Д04#2];\n4. Монитор предоставляется на 5 лет (если его состояние позволяет дальше работать) [Д05#1];\n5. Клиентские данные должны передаваться только через корпоративное хранилище [Д02#3]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q09", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 3.37, "prompt_tokens": 686, "completion_tokens": 118, "answer": "Второй монитор удалённому сотруднику выдаётся на основании заявки от руководителя, соответствующей категории «Оборудование» через сервис HelpDesk (Д05#1, 2.2). Резервный канал связи обязателен при работе с клиентскими проектами (Д04#2, 2.2). Заявка будет исполнена в течение 5 рабочих дней или 1 дня для критичных проектов. Клиентские данные следует передавать только через корпоративное хранилище (Д02#3, 3.2)."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q09", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 3.49, "prompt_tokens": 119, "completion_tokens": 116, "answer": "Для сотрудников компании «Вектор-Системы», работающих удалённо и нуждающихся во втором мониторе для клиентских проектов, действует регламент № 20-2023 «Предоставление дополнительного оборудования для удалённых сотрудников». Сотрудники должны подать заявление в IT-отдел через внутренний портал, указав обоснование необходимости дополнительного монитора. Заявки рассматриваются в течение 10 рабочих дней. Учёт ресурсов IT-инфраструктуры обязателен при одобрении заявки."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q09", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 2.99, "prompt_tokens": 119, "completion_tokens": 108, "answer": "Для сотрудников, работающих удалённо и нуждающихся в дополнительном мониторе для работы над клиентскими проектами, действует следующий регламент:\n\n1. Заявка на предоставление оборудования\n2. Согласование с непосредственным руководителем\n3. Одобрение ИТ-отдела\n4. Подписание акта приема-передачи\n\nНеобходимо учитывать следующие моменты:\n- Наличие свободного оборудования\n- Соответствие монитора рабочим требованиям\n- Признаки корпоративной политики использования ресурсов"}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q10", "condition": "A", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 5.6, "prompt_tokens": 750, "completion_tokens": 160, "answer": "Поездка в командировку должна быть оформлена заявкой в СЭД «Дирекум» не позднее чем за 5 рабочих дней до начала поездки [Д03#2, п. 2.1]. Перерасход на гостиницу более 10% от лимита возмещается только с письменной визой руководителя, однако конкретный лимит не указан [Д03#3, п. 3.3]. Авансовый отчёт после командировки необходимо сдавать в течение 3 рабочих дней после возвращения, с приложением всех соответствующих чеков, в том числе по гостинице [Д03#3, п. 3.1; Д03#3, п. 3.2]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q10", "condition": "A", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 3.99, "prompt_tokens": 750, "completion_tokens": 140, "answer": "Порядок действий сотрудника, отправляющегося в командировку в ООО «Вектор-Системы»:\n1. Заявку в СЭД «Дирекум» следует подавать не позднее чем за 5 рабочих дней до выезда [Д03#2.1].\n2. На гостиницу лимита расходов не установлено [Д03#2].\n3. Авансовый отчёт необходимо сдать не позднее, чем через 3 рабочих дня после возвращения [Д03#3.1].\n\nДанных о возможности тратить конкретную сумму на гостиницу в базе знаний нет. [Д03#2]"}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q10", "condition": "B", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 3.63, "prompt_tokens": 142, "completion_tokens": 121, "answer": "Сотрудникам «Вектор-Системы» необходимо подавать заявку на командировку в СЭД «Дирекум» не позднее чем за 3 рабочих дня до начала поездки. При утверждении командировки устанавливаются нормативы на проживание, которые зависят от должностей участников и продолжительности командировки. Нормы расходов можно найти во внутренних регламентах, раздел 11, п.3 «Порядок расходов на проживание и транспорт». Авансовый отчёт сдавать нужно в течение 5 рабочих дней после окончания командировки."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q10", "condition": "B", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 2.68, "prompt_tokens": 142, "completion_tokens": 96, "answer": "По внутренним регламентам \"Вектор-Системы\" заявку на командировку в СЭД «Дирекум» необходимо подавать не позднее чем за 5 рабочих дней до предполагаемой даты начала поездки. Сумма на гостиницу зависит от политики организации, но для сотрудников нашего уровня это обычно 6 500 рублей в сутки. Авансовый отчёт следует сдать в течение 3 рабочих дней после возвращения."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q11", "condition": "A", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 5.8, "prompt_tokens": 811, "completion_tokens": 164, "answer": "Данных по данному вопросу в базе знаний нет [Д03].\n\nОднако по договорам компания утверждает:\n1. Цикл согласования договора - не более 5 рабочих дней (Д10#2.1).\n2. Для корпоративных клиентов предусмотрены скидки при годовой оплате (Д08#1).\n\nЦена для тарифа Enterprise составляет 4900 рублей/месяц [Д08#1.3].\n3.480000 (рублей) = 4900 рублей * 12 месяцев.\n4. Для годовой оплаты действует скидка 10%: 480000 - 48000 = 432000 рублей."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q11", "condition": "A", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 3.01, "prompt_tokens": 811, "completion_tokens": 105, "answer": "Цикл согласования договора составляет не более 5 рабочих дней [Д10#2.1]. В фрагменте не указана цена тарифа \"Вектор-Офис Enterprise\" за годовой период и наличие скидок. Данные о ценах представлены только за месяц. \n\n[Д08#1] указывает тарифы на месяц, однако отсутствуют сведения о годовой скидке или стоимости. Таким образом, точную цену с учётом годовой скидки определить невозможно."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q11", "condition": "B", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 3.86, "prompt_tokens": 132, "completion_tokens": 129, "answer": "Данные по конкретным условиям договора на тариф Enterprise для корпоративных клиентов с годовой оплатой доступны в разделе «Коммерческое сотрудничество» на нашей внутренней платформе «Корпоратив-Портал».\n\nЦикл согласования договора занимает, в среднем, 5 рабочих дней, включая проверку правовой службы, финансового отдела и коммерческого директора.\n\nТочные условия тарифа, включая цену с учетом годовой скидки, определяются индивидуально для каждого клиента в зависимости от объема использования продукта. Обратитесь к менеджеру по работе с клиентами или к специалисту отдела продаж для получения точной информации."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q11", "condition": "B", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 3.29, "prompt_tokens": 132, "completion_tokens": 119, "answer": "Цикл согласования договора на тариф Enterprise, предусматривающий годовую оплату, занимает до двух недель с момента подачи всех необходимых документов клиентом в соответствующий отдел.\n\nСтоимость тарифа с учетом годовой оплаты может варьироваться в зависимости от объема предоставляемых услуг и уровня скидок, предоставляемых в рамках корпоративных программ и индивидуальных соглашений.\n\nТочную цену с учетом скидки за год можно получить, запросив коммерческое предложение у менеджера или отдела по работе с крупными клиентами. Данные по каждому конкретному клиенту индивидуальны."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q12", "condition": "A", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 1.36, "prompt_tokens": 594, "completion_tokens": 16, "answer": "Данных по размеру годовой премии сотрудников нет [Д01]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q12", "condition": "A", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.65, "prompt_tokens": 594, "completion_tokens": 20, "answer": "Данных по размеру годовой премии сотрудников в базе знаний нет. [Д01]"}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q12", "condition": "B", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 2.61, "prompt_tokens": 108, "completion_tokens": 84, "answer": "Годовая премия в «Вектор-Системах» зависит от индивидуальной оценки работы каждого сотрудника, а также от общей успешности компании за год. Премия может составлять до 30% от годового заработка и рассчитывается на основе системы премирования, регламентированной внутренним положением о премировании. Данные по конкретным сотрудникам сохраняются конфиденциально."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q12", "condition": "B", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 2.93, "prompt_tokens": 108, "completion_tokens": 106, "answer": "В компании \"Вектор-Системы\" размер годовой премии зависит от индивидуальной эффективности работы сотрудника, финансового положения компании, достижения установленных ключевых показателей эффективности (KPI) и выполнения поставленных задач. Конкретные критерии и проценты вознаграждения прописаны в системе мотивации и вознаграждения, утвержденной руководством компании.\n\nПодробнее это регулируется документом «Положение о системе премирования сотрудников», доступным во внутреннем портале для всех работников."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q13", "condition": "A", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.85, "prompt_tokens": 180, "completion_tokens": 17, "answer": "Данных по этому вопросу в базе знаний нет. [Нет источника]"}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q13", "condition": "A", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.38, "prompt_tokens": 180, "completion_tokens": 12, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q13", "condition": "B", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 3.39, "prompt_tokens": 110, "completion_tokens": 114, "answer": "Корпоративные SIM-карты в «Вектор-Системы» оформляются следующим образом:\n1. Совдение с HR-отделом для заполнения заявления на корпоративную SIM-карту.\n2. Проверка заявки руководителем подразделения.\n3. Передача документации в службу связи компании.\n4. Получение SIM-карты и PIN-кода через вашего руководителя или непосредственно из службы связи.\n\nДанные по конкретному сроку оформления доступны по запросу в HR-отделе."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q13", "condition": "B", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 3.01, "prompt_tokens": 110, "completion_tokens": 108, "answer": "Для оформления корпоративной SIM-карты сотрудникам «Вектор-Системы» необходимо обратиться в отдел информационных технологий (ИТ) с заявкой на получение SIM-карты. Заявку можно подать через внутренний портал компании в разделе «IT-поддержка» или лично посетить офис IT-отдела. После рассмотрения заявки и подтверждения соответствия сотрудника требованиям, SIM-карта оформляется в течение 3 рабочих дней и выдаётся в личном кабинете сотрудника."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q14", "condition": "A", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 1.63, "prompt_tokens": 734, "completion_tokens": 16, "answer": "Данных по этому вопросу в базе знаний нет [Д08]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q14", "condition": "A", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 2.57, "prompt_tokens": 734, "completion_tokens": 89, "answer": "Данных по тарифу «Вектор-Офис Premium» в базе знаний нет. В базе представлена линейка продуктов «Вектор-Офис», которая включает тарифы «Basic», «Standard» и «Enterprise» с ценами соответственно 1200, 2400 и 4900 рублей в месяц за пользователя [Д08#1]. [Д08]"}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q14", "condition": "B", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 2.47, "prompt_tokens": 114, "completion_tokens": 79, "answer": "Данные по тарифной сетке «Вектор-Офис» можно найти на сайте компании в разделе «Продукты и услуги» -> «Тарифные планы». Точную стоимость пакета Premium можно узнать после перехода по этой ссылке или через форму обратной связи на сайте. Также данные могут изменяться, рекомендуется уточнить актуальную цену через коммерческий отдел."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q14", "condition": "B", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 2.74, "prompt_tokens": 114, "completion_tokens": 99, "answer": "Данных по стоимости конкретного тарифа \"Вектор-Офис Premium\" в данный момент нет. Тарифные планы и цены на услуги компании «Вектор-Системы», включая \"Вектор-Офис\", могут варьироваться в зависимости от пакета услуг, дополнительных опций и условий контракта. Подробную информацию о стоимости и условиях каждого тарифного плана необходимо запросить в отделе продаж или в службу поддержки клиентов компании."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q15", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 1.85, "prompt_tokens": 599, "completion_tokens": 32, "answer": "Данных по процедуре оформления материнского капитала в базе знаний компании «Вектор-Системы» нет.\n\n[Д01]"}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q15", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 2.71, "prompt_tokens": 599, "completion_tokens": 95, "answer": "Данных по процедуре оформления материнского капитала в базе знаний нет. В регламенте оформления сотрудников ООО «Вектор-Системы» [Д01] описан только общий процесс приема на работу, оформление документов (трудовой договор, пропуск), испытательный срок и его оценка [Д01#1, Д01#2, Д01#3]. Вопросы материнского капитала в данном документе не освещаются."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q15", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 4.17, "prompt_tokens": 110, "completion_tokens": 142, "answer": "Данные по оформлению материнского капитала для сотрудников «Вектор-Системы» регулируются внутренним регламентом компании. Согласно разделу 3.5.1 «Социальные выплаты и льготы», оформление материнского капитала осуществляется на основании представленных сотрудником документов:\n\n1) заявления по форме №4;\n2) копий паспорта матери и ребенка;\n3) свидетельства о рождении ребенка.\n\nДокументы подаются в отдел кадров в течение 6 месяцев после рождения ребенка. Выплаты производятся ежемесячно на карту сотрудника. Подробности можно найти в Удостоверительной инструкции отдела кадров."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q15", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 2.47, "prompt_tokens": 110, "completion_tokens": 88, "answer": "Данные по конкретной процедуре оформления материнского капитала в компании «Вектор-Системы» не содержатся во внутренних регламентах. Однако, согласно общим политикам по трудоустройству и поддержке сотрудников, родителям, имеющим детей, предоставляется возможность использовать данный государственное пособие (материнский капитал) на определенных условиях. Детали можно уточнить в отделе кадров."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q16", "condition": "A", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 2.15, "prompt_tokens": 423, "completion_tokens": 53, "answer": "Данных по этому вопросу в базе знаний нет. [Д04] — регламент удаленной работы, не упоминает расположение столовой и меню. [Д10] регулирует документооборот и не содержит данных о столовой."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q16", "condition": "A", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.42, "prompt_tokens": 423, "completion_tokens": 12, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q16", "condition": "B", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 3.56, "prompt_tokens": 113, "completion_tokens": 120, "answer": "Столовая находится на 3 этаже, в восточном крыле здания. Меню обновляется ежедневно. В будние дни можно насладиться разнообразной домашней едой, включая первые блюда, гарниры, мясные и овощные блюда, салаты, а также десерты и напитки. Меню включает как традиционные, так и сезонные блюда, учитывая предпочтения сотрудников. Еженедельно публикуется расписание меню, доступное на информационном стенде около лифта и в электронной форме на внутреннем портале компании."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q16", "condition": "B", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 1.9, "prompt_tokens": 113, "completion_tokens": 67, "answer": "В нашем офисе столовая располагается на третьем этаже, в западном крыле здания. Документ о расписании работы столовой, меню и ценах можно найти на информационном стенде в приемной второго этажа или в электронной внутренней системе, разделе \"Пользовательские услуги\"."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q17", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 2.12, "prompt_tokens": 710, "completion_tokens": 34, "answer": "Нет, согласно политике ООО «Вектор-Системы» ротация пароля происходит каждые 90 дней [Д02#1]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q17", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 1.76, "prompt_tokens": 710, "completion_tokens": 60, "answer": "Пароли в ООО «Вектор-Системы» должны меняться каждые 90 дней [Д02#1.1.2]. Таким образом, утверждение о необходимости смены пароля каждые 60 дней не соответствует установленным правилам компании."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q17", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 2.87, "prompt_tokens": 116, "completion_tokens": 94, "answer": "Пароль для доступа к корпоративным системам должен регулярно обновляться, согласно внутреннему регламенту безопасности. Данные по этому вопросу подтверждают, что пароль необходимо менять каждые 90 дней. Дополнительно, он должен содержать не менее 8 символов, включать цифры, буквы в верхнем и нижнем регистре, и специальные символы. Рекомендуется также устанавливать разные пароли для различных сервисов."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q17", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 2.45, "prompt_tokens": 116, "completion_tokens": 88, "answer": "Да, согласно регламенту безопасности «Вектор-Системы», все сотрудники компании обязаны регулярно менять свои пароли для доступа к корпоративным системам. Внутренний регламент № БЗ-02/23 устанавливает, что пароль необходимо изменять каждые 90 дней. Это требование направлено на защиту конфиденциальной информации и минимизацию риска несанкционированного доступа."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q18", "condition": "A", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 4.39, "prompt_tokens": 648, "completion_tokens": 123, "answer": "По регламенту ООО «Вектор-Системы» ежегодный основной отпуск составляет 28 календарных дней, и данный регламент (Д09#1) не предусматривает возможности разделения этого отпуска на однодневные части. Дополнительные дни отпуска (Д09#1) также не имеют условий о разбиении на короткие периоды.\n\nЕсли есть особые обстоятельства, требующие частичного использования отпуска, сотруднику следует обратиться к своему руководителю или HR-отделу для обсуждения индивидуальных условий [Д09#1]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q18", "condition": "A", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 1.13, "prompt_tokens": 648, "completion_tokens": 37, "answer": "Данных по возможности разделения годового отпуска на однодневные части в базе знаний ООО «Вектор-Системы» нет [Д09#1]."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q18", "condition": "B", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 3.66, "prompt_tokens": 113, "completion_tokens": 123, "answer": "Да, согласно регламентам компании \"Вектор-Системы\", сотрудники могут разделить свой ежегодный основной оплачиваемый отпуск на несколько периодов. Однако необходимо соблюдение некоторых условий:\n\n1. Каждый период отпуска должен составлять не менее 3 календарных дней.\n2. Всего можно взять максимум 4 однодневных фрагмента из 14 календарных дней отпуска.\n3. При необходимости деления отпуска следует согласовать план с руководителем отдела.\n\nВсе подробности можно найти в локальном акте \"Условия предоставления отпусков\"."}
{"model": "M8-saiga_gemma3_12b-Q8_0", "question": "Q18", "condition": "B", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 2.24, "prompt_tokens": 113, "completion_tokens": 80, "answer": "Согласно внутреннему регламенту компании «Вектор-Системы», разбить ежегодный основной отпуск не более двух раза на части, при этом каждая часть должна составлять не менее пяти календарных дней. Возможно разделение ипотечного/материнства/семейного отпуска. Подробнее ознакомиться с правилами можно в локальном нормативном акте."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q01", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 12.82, "prompt_tokens": 740, "completion_tokens": 400, "answer": "Испытательный срок составляет 3 месяца для обычного сотрудника и 6 месяцев для руководителей подразделений [Д01"}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q01", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 11.59, "prompt_tokens": 740, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q01", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 7.6, "prompt_tokens": 122, "completion_tokens": 256, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q01", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 122, "completion_tokens": 241, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q02", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 12.18, "prompt_tokens": 430, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q02", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 11.48, "prompt_tokens": 430, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q02", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 9.62, "prompt_tokens": 119, "completion_tokens": 327, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q02", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 6.96, "prompt_tokens": 119, "completion_tokens": 244, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q03", "condition": "A", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 12.75, "prompt_tokens": 724, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q03", "condition": "A", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 11.57, "prompt_tokens": 724, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q03", "condition": "B", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 8.27, "prompt_tokens": 120, "completion_tokens": 279, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q03", "condition": "B", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 9.14, "prompt_tokens": 120, "completion_tokens": 321, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q04", "condition": "A", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 12.76, "prompt_tokens": 745, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q04", "condition": "A", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 11.58, "prompt_tokens": 745, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q04", "condition": "B", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 8.75, "prompt_tokens": 127, "completion_tokens": 296, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q04", "condition": "B", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 6.03, "prompt_tokens": 127, "completion_tokens": 211, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q05", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 12.07, "prompt_tokens": 393, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q05", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 11.52, "prompt_tokens": 393, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q05", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 9.98, "prompt_tokens": 123, "completion_tokens": 340, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q05", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.04, "prompt_tokens": 123, "completion_tokens": 247, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q06", "condition": "A", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 12.67, "prompt_tokens": 681, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q06", "condition": "A", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 11.55, "prompt_tokens": 681, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q06", "condition": "B", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 8.83, "prompt_tokens": 120, "completion_tokens": 299, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q06", "condition": "B", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 7.26, "prompt_tokens": 120, "completion_tokens": 255, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q07", "condition": "A", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 12.77, "prompt_tokens": 769, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q07", "condition": "A", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 11.57, "prompt_tokens": 769, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q07", "condition": "B", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 7.28, "prompt_tokens": 132, "completion_tokens": 244, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q07", "condition": "B", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 8.8, "prompt_tokens": 132, "completion_tokens": 309, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q08", "condition": "A", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 12.57, "prompt_tokens": 663, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q08", "condition": "A", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 11.54, "prompt_tokens": 663, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q08", "condition": "B", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 9.38, "prompt_tokens": 121, "completion_tokens": 318, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q08", "condition": "B", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 7.86, "prompt_tokens": 121, "completion_tokens": 276, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q09", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 12.69, "prompt_tokens": 693, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q09", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 11.55, "prompt_tokens": 693, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q09", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 8.4, "prompt_tokens": 126, "completion_tokens": 284, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q09", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 8.52, "prompt_tokens": 126, "completion_tokens": 299, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q10", "condition": "A", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 12.77, "prompt_tokens": 757, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q10", "condition": "A", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 11.58, "prompt_tokens": 757, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q10", "condition": "B", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 8.65, "prompt_tokens": 149, "completion_tokens": 292, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q10", "condition": "B", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 7.91, "prompt_tokens": 149, "completion_tokens": 277, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q11", "condition": "A", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 12.89, "prompt_tokens": 818, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q11", "condition": "A", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 11.59, "prompt_tokens": 818, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q11", "condition": "B", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 9.03, "prompt_tokens": 139, "completion_tokens": 306, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q11", "condition": "B", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 7.42, "prompt_tokens": 139, "completion_tokens": 260, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q12", "condition": "A", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 12.43, "prompt_tokens": 601, "completion_tokens": 400, "answer": "Данных по этому вопросу в"}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q12", "condition": "A", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 10.56, "prompt_tokens": 601, "completion_tokens": 367, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q12", "condition": "B", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 8.04, "prompt_tokens": 115, "completion_tokens": 271, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q12", "condition": "B", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 9.4, "prompt_tokens": 115, "completion_tokens": 330, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q13", "condition": "A", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 6.41, "prompt_tokens": 187, "completion_tokens": 211, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q13", "condition": "A", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 7.31, "prompt_tokens": 187, "completion_tokens": 255, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q13", "condition": "B", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 8.4, "prompt_tokens": 117, "completion_tokens": 285, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q13", "condition": "B", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 6.34, "prompt_tokens": 117, "completion_tokens": 222, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q14", "condition": "A", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 12.77, "prompt_tokens": 741, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q14", "condition": "A", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 10.94, "prompt_tokens": 741, "completion_tokens": 378, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q14", "condition": "B", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 7.26, "prompt_tokens": 121, "completion_tokens": 243, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q14", "condition": "B", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 7.39, "prompt_tokens": 121, "completion_tokens": 259, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q15", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 11.14, "prompt_tokens": 606, "completion_tokens": 353, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q15", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 10.68, "prompt_tokens": 606, "completion_tokens": 371, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q15", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 8.83, "prompt_tokens": 117, "completion_tokens": 299, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q15", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 9.76, "prompt_tokens": 117, "completion_tokens": 343, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q16", "condition": "A", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 10.93, "prompt_tokens": 430, "completion_tokens": 355, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q16", "condition": "A", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 11.43, "prompt_tokens": 430, "completion_tokens": 398, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q16", "condition": "B", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 5.84, "prompt_tokens": 120, "completion_tokens": 194, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q16", "condition": "B", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.87, "prompt_tokens": 120, "completion_tokens": 276, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q17", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 12.78, "prompt_tokens": 717, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q17", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 11.57, "prompt_tokens": 717, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q17", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 10.03, "prompt_tokens": 123, "completion_tokens": 341, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q17", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 9.46, "prompt_tokens": 123, "completion_tokens": 332, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q18", "condition": "A", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 10.31, "prompt_tokens": 655, "completion_tokens": 322, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q18", "condition": "A", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 11.53, "prompt_tokens": 655, "completion_tokens": 400, "answer": ""}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q18", "condition": "B", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 8.82, "prompt_tokens": 120, "completion_tokens": 299, "answer": "Данных по этому вопросу нет."}
{"model": "M9-gemma-4-12b-it-Q8_0", "question": "Q18", "condition": "B", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 9.2, "prompt_tokens": 120, "completion_tokens": 323, "answer": "Данных по этому вопросу нет."}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q01", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 8.9, "prompt_tokens": 755, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q01", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 7.58, "prompt_tokens": 755, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q01", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 6.78, "prompt_tokens": 113, "completion_tokens": 359, "answer": "Данных по этому вопросу нет"}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q01", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 6.26, "prompt_tokens": 113, "completion_tokens": 342, "answer": "Данных по этому вопросу нет."}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q02", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.95, "prompt_tokens": 431, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q02", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.53, "prompt_tokens": 431, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q02", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.52, "prompt_tokens": 109, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q02", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.31, "prompt_tokens": 109, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q03", "condition": "A", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 8.37, "prompt_tokens": 714, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q03", "condition": "A", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 7.56, "prompt_tokens": 714, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q03", "condition": "B", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 7.53, "prompt_tokens": 111, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q03", "condition": "B", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 7.3, "prompt_tokens": 111, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q04", "condition": "A", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 8.41, "prompt_tokens": 740, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q04", "condition": "A", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 7.57, "prompt_tokens": 740, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q04", "condition": "B", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 7.56, "prompt_tokens": 117, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q04", "condition": "B", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 5.54, "prompt_tokens": 117, "completion_tokens": 301, "answer": "Данных по этому вопросу нет"}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q05", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.89, "prompt_tokens": 384, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q05", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.51, "prompt_tokens": 384, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q05", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.53, "prompt_tokens": 109, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q05", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.31, "prompt_tokens": 109, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q06", "condition": "A", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 8.3, "prompt_tokens": 677, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q06", "condition": "A", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 7.58, "prompt_tokens": 677, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q06", "condition": "B", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 6.76, "prompt_tokens": 109, "completion_tokens": 357, "answer": "Данных по этому вопросу нет."}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q06", "condition": "B", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 6.46, "prompt_tokens": 109, "completion_tokens": 353, "answer": "Данных по этому вопросу нет."}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q07", "condition": "A", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 8.36, "prompt_tokens": 726, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q07", "condition": "A", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 7.57, "prompt_tokens": 726, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q07", "condition": "B", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 7.56, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q07", "condition": "B", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 7.36, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q08", "condition": "A", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 8.36, "prompt_tokens": 660, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q08", "condition": "A", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 7.57, "prompt_tokens": 660, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q08", "condition": "B", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 7.51, "prompt_tokens": 113, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q08", "condition": "B", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 7.27, "prompt_tokens": 113, "completion_tokens": 400, "answer": "Данных по этому вопросу нет"}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q09", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 8.0, "prompt_tokens": 681, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q09", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 7.58, "prompt_tokens": 681, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q09", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 7.55, "prompt_tokens": 114, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q09", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 7.32, "prompt_tokens": 114, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q10", "condition": "A", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 8.49, "prompt_tokens": 747, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q10", "condition": "A", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 7.57, "prompt_tokens": 747, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q10", "condition": "B", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 7.57, "prompt_tokens": 138, "completion_tokens": 398, "answer": "Данных по этому вопросу нет"}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q10", "condition": "B", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 7.37, "prompt_tokens": 138, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q11", "condition": "A", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 8.46, "prompt_tokens": 810, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q11", "condition": "A", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 7.52, "prompt_tokens": 810, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q11", "condition": "B", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 7.59, "prompt_tokens": 127, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q11", "condition": "B", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 6.14, "prompt_tokens": 127, "completion_tokens": 334, "answer": "Данных по этому вопросу нет"}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q12", "condition": "A", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 7.94, "prompt_tokens": 541, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q12", "condition": "A", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 7.27, "prompt_tokens": 541, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q12", "condition": "B", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 7.46, "prompt_tokens": 104, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q12", "condition": "B", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 7.3, "prompt_tokens": 104, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q13", "condition": "A", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 7.64, "prompt_tokens": 182, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q13", "condition": "A", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 7.4, "prompt_tokens": 182, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q13", "condition": "B", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 7.48, "prompt_tokens": 106, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q13", "condition": "B", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 7.3, "prompt_tokens": 106, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q14", "condition": "A", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 8.24, "prompt_tokens": 690, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q14", "condition": "A", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 7.58, "prompt_tokens": 690, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q14", "condition": "B", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 7.54, "prompt_tokens": 109, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q14", "condition": "B", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 7.28, "prompt_tokens": 109, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q15", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 7.9, "prompt_tokens": 589, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q15", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 7.53, "prompt_tokens": 589, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q15", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 7.49, "prompt_tokens": 105, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q15", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 7.29, "prompt_tokens": 105, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q16", "condition": "A", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.82, "prompt_tokens": 418, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q16", "condition": "A", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.53, "prompt_tokens": 418, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q16", "condition": "B", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.53, "prompt_tokens": 108, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q16", "condition": "B", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 6.84, "prompt_tokens": 108, "completion_tokens": 373, "answer": "Данных по этому вопросу нет."}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q17", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 8.01, "prompt_tokens": 704, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q17", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 7.58, "prompt_tokens": 704, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q17", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 7.52, "prompt_tokens": 110, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q17", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 7.31, "prompt_tokens": 110, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q18", "condition": "A", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 8.2, "prompt_tokens": 615, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q18", "condition": "A", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 7.52, "prompt_tokens": 615, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q18", "condition": "B", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 7.52, "prompt_tokens": 110, "completion_tokens": 400, "answer": ""}
{"model": "M10-GLM-4.7-Flash-Q6_K", "question": "Q18", "condition": "B", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 7.29, "prompt_tokens": 110, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q01", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 27.35, "prompt_tokens": 740, "completion_tokens": 375, "answer": "Испытательный срок для сотрудников составляет 3 месяца, а для руководителей подразделений — 6 месяцев [Д01]."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q01", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 23.5, "prompt_tokens": 740, "completion_tokens": 362, "answer": "Испытательный срок для сотрудников составляет 3 месяца, а для руководителей подразделений — 6 месяцев [Д01]."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q01", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 12.8, "prompt_tokens": 122, "completion_tokens": 189, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q01", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 12.99, "prompt_tokens": 122, "completion_tokens": 203, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q02", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 23.31, "prompt_tokens": 430, "completion_tokens": 337, "answer": "Пароль необходимо менять каждые 90 дней [Д02]. Минимальная длина пароля — 12 знаков [Д02]."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q02", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 20.81, "prompt_tokens": 430, "completion_tokens": 323, "answer": "Минимальная длина пароля — 12 знаков [Д02]. Ротация пароля производится каждые 90 дней [Д02]."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q02", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 14.55, "prompt_tokens": 119, "completion_tokens": 217, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q02", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 15.27, "prompt_tokens": 119, "completion_tokens": 239, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q03", "condition": "A", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 28.77, "prompt_tokens": 724, "completion_tokens": 400, "answer": "Суточные составляют 1500 рублей за каждый день поездки, включая"}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q03", "condition": "A", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 25.97, "prompt_tokens": 724, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q03", "condition": "B", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 14.82, "prompt_tokens": 120, "completion_tokens": 220, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q03", "condition": "B", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 14.13, "prompt_tokens": 120, "completion_tokens": 225, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q04", "condition": "A", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 28.85, "prompt_tokens": 745, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q04", "condition": "A", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 25.51, "prompt_tokens": 745, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q04", "condition": "B", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 16.65, "prompt_tokens": 127, "completion_tokens": 253, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q04", "condition": "B", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 15.03, "prompt_tokens": 127, "completion_tokens": 239, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q05", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 26.75, "prompt_tokens": 393, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q05", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 25.35, "prompt_tokens": 393, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q05", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 14.35, "prompt_tokens": 123, "completion_tokens": 217, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q05", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 12.11, "prompt_tokens": 123, "completion_tokens": 192, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q06", "condition": "A", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 28.18, "prompt_tokens": 681, "completion_tokens": 400, "answer": "Для заявок Critical время реакции составляет 30 минут, а время решения — 4 часа [Д06]. Для заявок Major время реакции — 2 часа, время решения — 1 рабочий день [Д06]."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q06", "condition": "A", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 25.49, "prompt_tokens": 681, "completion_tokens": 400, "answer": "Для заявок Critical время реакции составляет 30 минут,"}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q06", "condition": "B", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 15.52, "prompt_tokens": 120, "completion_tokens": 235, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q06", "condition": "B", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 14.23, "prompt_tokens": 120, "completion_tokens": 226, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q07", "condition": "A", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 28.42, "prompt_tokens": 769, "completion_tokens": 400, "answer": "Тариф «Вектор-Офис Standard» стоит 2400 рублей за пользователя в"}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q07", "condition": "A", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 24.48, "prompt_tokens": 769, "completion_tokens": 383, "answer": "Тариф «Вектор-Офис Standard» стоит 2400 рублей за пользователя в месяц [Д08]. При годовой оплате предоставляется скидка 10 % от тарифа [Д08]."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q07", "condition": "B", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 13.52, "prompt_tokens": 132, "completion_tokens": 202, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q07", "condition": "B", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 14.68, "prompt_tokens": 132, "completion_tokens": 233, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q08", "condition": "A", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 28.06, "prompt_tokens": 663, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q08", "condition": "A", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 25.5, "prompt_tokens": 663, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q08", "condition": "B", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 15.44, "prompt_tokens": 121, "completion_tokens": 233, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q08", "condition": "B", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 12.73, "prompt_tokens": 121, "completion_tokens": 202, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q09", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 28.25, "prompt_tokens": 693, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q09", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 25.51, "prompt_tokens": 693, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q09", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 17.44, "prompt_tokens": 126, "completion_tokens": 265, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q09", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 14.25, "prompt_tokens": 126, "completion_tokens": 226, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q10", "condition": "A", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 28.43, "prompt_tokens": 757, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q10", "condition": "A", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 25.55, "prompt_tokens": 757, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q10", "condition": "B", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 17.24, "prompt_tokens": 149, "completion_tokens": 260, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q10", "condition": "B", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 17.08, "prompt_tokens": 149, "completion_tokens": 271, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q11", "condition": "A", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 28.68, "prompt_tokens": 818, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q11", "condition": "A", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 25.56, "prompt_tokens": 818, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q11", "condition": "B", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 16.96, "prompt_tokens": 139, "completion_tokens": 257, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q11", "condition": "B", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 13.37, "prompt_tokens": 139, "completion_tokens": 212, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q12", "condition": "A", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 27.67, "prompt_tokens": 601, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q12", "condition": "A", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 25.4, "prompt_tokens": 601, "completion_tokens": 400, "answer": ""}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q12", "condition": "B", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 15.03, "prompt_tokens": 115, "completion_tokens": 227, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q12", "condition": "B", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 17.23, "prompt_tokens": 115, "completion_tokens": 274, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q13", "condition": "A", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 16.43, "prompt_tokens": 187, "completion_tokens": 247, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q13", "condition": "A", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 14.94, "prompt_tokens": 187, "completion_tokens": 236, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q13", "condition": "B", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 15.11, "prompt_tokens": 117, "completion_tokens": 229, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q13", "condition": "B", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 13.55, "prompt_tokens": 117, "completion_tokens": 215, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q14", "condition": "A", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 28.06, "prompt_tokens": 741, "completion_tokens": 394, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q14", "condition": "A", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 24.15, "prompt_tokens": 741, "completion_tokens": 378, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q14", "condition": "B", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 15.32, "prompt_tokens": 121, "completion_tokens": 231, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q14", "condition": "B", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 15.8, "prompt_tokens": 121, "completion_tokens": 251, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q15", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 20.75, "prompt_tokens": 606, "completion_tokens": 288, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q15", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 20.76, "prompt_tokens": 606, "completion_tokens": 326, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q15", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 19.89, "prompt_tokens": 117, "completion_tokens": 305, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q15", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 18.62, "prompt_tokens": 117, "completion_tokens": 296, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q16", "condition": "A", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 19.79, "prompt_tokens": 430, "completion_tokens": 286, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q16", "condition": "A", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 21.79, "prompt_tokens": 430, "completion_tokens": 344, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q16", "condition": "B", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 16.15, "prompt_tokens": 120, "completion_tokens": 245, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q16", "condition": "B", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 8.6, "prompt_tokens": 120, "completion_tokens": 136, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q17", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 22.2, "prompt_tokens": 717, "completion_tokens": 303, "answer": "Нет, ротация пароля производится каждые 90 дней [Д02]."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q17", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 21.17, "prompt_tokens": 717, "completion_tokens": 331, "answer": "Нет, ротация пароля осуществляется каждые 90 дней [Д02#1]."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q17", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 16.33, "prompt_tokens": 123, "completion_tokens": 247, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q17", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 14.8, "prompt_tokens": 123, "completion_tokens": 235, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q18", "condition": "A", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 24.59, "prompt_tokens": 655, "completion_tokens": 346, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q18", "condition": "A", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 22.54, "prompt_tokens": 655, "completion_tokens": 354, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q18", "condition": "B", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 16.59, "prompt_tokens": 120, "completion_tokens": 251, "answer": "Данных по этому вопросу нет."}
{"model": "M11-gemma-4-31B-it-Q6_K", "question": "Q18", "condition": "B", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 15.93, "prompt_tokens": 120, "completion_tokens": 253, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q01", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 4.16, "prompt_tokens": 670, "completion_tokens": 25, "answer": "Испытательный срок для обычного сотрудника — 3 месяца, для руководителей подразделений — 6 месяцев [Д01]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q01", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 1.48, "prompt_tokens": 670, "completion_tokens": 25, "answer": "Испытательный срок для обычного сотрудника — 3 месяца, для руководителей подразделений — 6 месяцев [Д01]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q01", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 0.97, "prompt_tokens": 110, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q01", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 110, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q02", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 3.46, "prompt_tokens": 386, "completion_tokens": 35, "answer": "Пароль нужно менять каждые 90 дней, минимальная длина — 12 знаков с обязательными цифрами и спецсимволами [Д02]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q02", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 2.04, "prompt_tokens": 386, "completion_tokens": 35, "answer": "Пароль нужно менять каждые 90 дней, минимальная длина — 12 знаков с обязательными цифрами и спецсимволами [Д02]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q02", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.71, "prompt_tokens": 110, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q02", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 110, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q03", "condition": "A", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 6.0, "prompt_tokens": 660, "completion_tokens": 66, "answer": "Суточные — 1500 рублей за каждый день поездки, включая дни в пути [Д03#1]. Лимит на гостиницу — не дороже 6500 рублей за ночь; для городов присутствия клиента — до 8000 рублей по согласованию с руководителем [Д03#1]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q03", "condition": "A", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 3.9, "prompt_tokens": 660, "completion_tokens": 66, "answer": "Суточные — 1500 рублей за каждый день поездки, включая дни в пути [Д03#1]. Лимит на гостиницу — не дороже 6500 рублей за ночь; для городов присутствия клиента — до 8000 рублей по согласованию с руководителем [Д03#1]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q03", "condition": "B", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 0.71, "prompt_tokens": 110, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q03", "condition": "B", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 0.47, "prompt_tokens": 110, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q04", "condition": "A", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 6.18, "prompt_tokens": 689, "completion_tokens": 61, "answer": "Удалёнщик обязан быть в офисе не менее 2 рабочих дней в неделю [Д04#1]. Ядро доступности — с 11:00 до 15:00, в это время сотрудник должен быть доступен по видео и телефону [Д04#1]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q04", "condition": "A", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 3.6, "prompt_tokens": 689, "completion_tokens": 61, "answer": "Удалёнщик обязан быть в офисе не менее 2 рабочих дней в неделю [Д04#1]. Ядро доступности — с 11:00 до 15:00, в это время сотрудник должен быть доступен по видео и телефону [Д04#1]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q04", "condition": "B", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 0.71, "prompt_tokens": 116, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q04", "condition": "B", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 0.47, "prompt_tokens": 116, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q05", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 5.78, "prompt_tokens": 356, "completion_tokens": 75, "answer": "Ноутбук обновляется каждые 3 года с даты выдачи [Д05#1], монитор — каждые 5 лет [Д05#1]. Заявку на второй монитор подаёт руководитель [Д05#1]. Данных о подаче заявок на обновление основного монитора или ноутбука в базе нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q05", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 3.13, "prompt_tokens": 356, "completion_tokens": 54, "answer": "Ноутбук обновляется каждые 3 года с даты выдачи [Д05#1], монитор — каждые 5 лет [Д05#1]. Заявку на второй монитор подаёт руководитель [Д05#1]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q05", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.7, "prompt_tokens": 111, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q05", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 111, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q06", "condition": "A", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 4.74, "prompt_tokens": 628, "completion_tokens": 46, "answer": "Для Critical: реакция — 30 минут, решение — 4 часа [Д06#2].  \nДля Major: реакция — 2 часа, решение — 1 рабочий день [Д06#2]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q06", "condition": "A", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 2.7, "prompt_tokens": 628, "completion_tokens": 46, "answer": "Для Critical: реакция — 30 минут, решение — 4 часа [Д06#2].  \nДля Major: реакция — 2 часа, решение — 1 рабочий день [Д06#2]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q06", "condition": "B", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.7, "prompt_tokens": 110, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q06", "condition": "B", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 110, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q07", "condition": "A", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 5.52, "prompt_tokens": 709, "completion_tokens": 50, "answer": "Тариф «Вектор-Офис Standard» стоит 2400 рублей за пользователя в месяц [Д08#1]. При годовой оплате предоставляется скидка 10 % от тарифа [Д08#2]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q07", "condition": "A", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 2.94, "prompt_tokens": 709, "completion_tokens": 50, "answer": "Тариф «Вектор-Офис Standard» стоит 2400 рублей за пользователя в месяц [Д08#1]. При годовой оплате предоставляется скидка 10 % от тарифа [Д08#2]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q07", "condition": "B", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 0.71, "prompt_tokens": 120, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q07", "condition": "B", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 120, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q08", "condition": "A", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 5.26, "prompt_tokens": 603, "completion_tokens": 51, "answer": "Ежегодный отпуск — 28 календарных дней [Д09#1]. Заявление подаётся в СЭД «Дирекум» не позднее чем за 2 недели до начала отпуска [Д09#3]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q08", "condition": "A", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 3.0, "prompt_tokens": 603, "completion_tokens": 51, "answer": "Ежегодный отпуск — 28 календарных дней [Д09#1]. Заявление подаётся в СЭД «Дирекум» не позднее чем за 2 недели до начала отпуска [Д09#3]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q08", "condition": "B", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 0.71, "prompt_tokens": 112, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q08", "condition": "B", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 112, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q09", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 6.72, "prompt_tokens": 642, "completion_tokens": 72, "answer": "Второй монитор выдаётся по заявке руководителя [Д05#1]. Заявку подаёт через HelpDesk, категория «Оборудование» [Д05#2]. Срок исполнения — до 5 рабочих дней. При работе с клиентскими данными использовать только корпоративное оборудование, личное — запрещено [Д04#2]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q09", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 4.23, "prompt_tokens": 642, "completion_tokens": 72, "answer": "Второй монитор выдаётся по заявке руководителя [Д05#1]. Заявку подаёт через HelpDesk, категория «Оборудование» [Д05#2]. Срок исполнения — до 5 рабочих дней. При работе с клиентскими данными использовать только корпоративное оборудование, личное — запрещено [Д04#2]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q09", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.71, "prompt_tokens": 116, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q09", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 116, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q10", "condition": "A", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 6.52, "prompt_tokens": 683, "completion_tokens": 67, "answer": "Заявку в СЭД «Дирекум» подавайте не позднее чем за 5 рабочих дней до выезда [Д03#2]. Лимит расходов на гостиницу не указан в базе знаний. Авансовый отчёт сдавайте в течение 3 рабочих дней после возвращения [Д03#3]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q10", "condition": "A", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 3.94, "prompt_tokens": 683, "completion_tokens": 67, "answer": "Заявку в СЭД «Дирекум» подавайте не позднее чем за 5 рабочих дней до выезда [Д03#2]. Лимит расходов на гостиницу не указан в базе знаний. Авансовый отчёт сдавайте в течение 3 рабочих дней после возвращения [Д03#3]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q10", "condition": "B", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 0.77, "prompt_tokens": 132, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q10", "condition": "B", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 132, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q11", "condition": "A", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 7.12, "prompt_tokens": 738, "completion_tokens": 72, "answer": "Цикл согласования договора — не более 5 рабочих дней [Д10#2]. Цена тарифа «Вектор-Офис Enterprise» — 4900 руб./мес., годовая оплата: 58 800 руб. (без скидки, в документах скидки не указаны) [Д08#1]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q11", "condition": "A", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 4.42, "prompt_tokens": 738, "completion_tokens": 74, "answer": "Цикл согласования договора — не более 5 рабочих дней [Д10#2]. Цена тарифа «Вектор-Офис Enterprise» — 4900 руб./мес., годовая оплата без скидки — 58 800 руб. Информация о скидках за годовую оплату в базе отсутствует [Д08#1]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q11", "condition": "B", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 0.7, "prompt_tokens": 126, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q11", "condition": "B", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 126, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q12", "condition": "A", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 2.76, "prompt_tokens": 544, "completion_tokens": 11, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q12", "condition": "A", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.65, "prompt_tokens": 544, "completion_tokens": 11, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q12", "condition": "B", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.7, "prompt_tokens": 107, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q12", "condition": "B", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.47, "prompt_tokens": 107, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q13", "condition": "A", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 1.25, "prompt_tokens": 165, "completion_tokens": 11, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q13", "condition": "A", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.63, "prompt_tokens": 165, "completion_tokens": 11, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q13", "condition": "B", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.68, "prompt_tokens": 108, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q13", "condition": "B", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 108, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q14", "condition": "A", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 2.86, "prompt_tokens": 680, "completion_tokens": 11, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q14", "condition": "A", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.65, "prompt_tokens": 680, "completion_tokens": 11, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q14", "condition": "B", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.71, "prompt_tokens": 111, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q14", "condition": "B", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 111, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q15", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 2.0, "prompt_tokens": 551, "completion_tokens": 11, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q15", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.66, "prompt_tokens": 551, "completion_tokens": 11, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q15", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.7, "prompt_tokens": 108, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q15", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 108, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q16", "condition": "A", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 1.95, "prompt_tokens": 393, "completion_tokens": 11, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q16", "condition": "A", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.64, "prompt_tokens": 393, "completion_tokens": 11, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q16", "condition": "B", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.7, "prompt_tokens": 109, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q16", "condition": "B", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 109, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q17", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 3.15, "prompt_tokens": 658, "completion_tokens": 18, "answer": "Нет, пароль нужно менять каждые 90 дней [Д02]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q17", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 1.06, "prompt_tokens": 658, "completion_tokens": 18, "answer": "Нет, пароль нужно менять каждые 90 дней [Д02]."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q17", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 0.71, "prompt_tokens": 115, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q17", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 0.47, "prompt_tokens": 115, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q18", "condition": "A", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 2.95, "prompt_tokens": 591, "completion_tokens": 11, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q18", "condition": "A", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.65, "prompt_tokens": 591, "completion_tokens": 11, "answer": "Данных по этому вопросу в базе знаний нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q18", "condition": "B", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.7, "prompt_tokens": 111, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M12-T-pro-it-2.1-Q6_K", "question": "Q18", "condition": "B", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.46, "prompt_tokens": 111, "completion_tokens": 8, "answer": "Данных по этому вопросу нет."}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q01", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 7.97, "prompt_tokens": 725, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q01", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 6.89, "prompt_tokens": 725, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q01", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 7.08, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q01", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 6.87, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q02", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.19, "prompt_tokens": 433, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q02", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 433, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q02", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.09, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q02", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q03", "condition": "A", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 7.52, "prompt_tokens": 735, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q03", "condition": "A", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 6.9, "prompt_tokens": 735, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q03", "condition": "B", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 7.11, "prompt_tokens": 124, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q03", "condition": "B", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 124, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q04", "condition": "A", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 7.53, "prompt_tokens": 751, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q04", "condition": "A", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 6.95, "prompt_tokens": 751, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q04", "condition": "B", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 7.09, "prompt_tokens": 126, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q04", "condition": "B", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 6.85, "prompt_tokens": 126, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q05", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.19, "prompt_tokens": 395, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q05", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 6.9, "prompt_tokens": 395, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q05", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.04, "prompt_tokens": 122, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q05", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 122, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q06", "condition": "A", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 7.49, "prompt_tokens": 680, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q06", "condition": "A", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 6.9, "prompt_tokens": 680, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q06", "condition": "B", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 7.09, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q06", "condition": "B", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q07", "condition": "A", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 7.53, "prompt_tokens": 765, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q07", "condition": "A", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 6.96, "prompt_tokens": 765, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q07", "condition": "B", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 7.1, "prompt_tokens": 130, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q07", "condition": "B", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 6.9, "prompt_tokens": 130, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q08", "condition": "A", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 7.4, "prompt_tokens": 651, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q08", "condition": "A", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 6.87, "prompt_tokens": 651, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q08", "condition": "B", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 7.07, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q08", "condition": "B", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q09", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 7.5, "prompt_tokens": 692, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q09", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 6.86, "prompt_tokens": 692, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q09", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 7.11, "prompt_tokens": 125, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q09", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 6.86, "prompt_tokens": 125, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q10", "condition": "A", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 7.49, "prompt_tokens": 759, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q10", "condition": "A", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 6.9, "prompt_tokens": 759, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q10", "condition": "B", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 7.14, "prompt_tokens": 149, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q10", "condition": "B", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 149, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q11", "condition": "A", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 7.53, "prompt_tokens": 825, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q11", "condition": "A", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 825, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q11", "condition": "B", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 7.08, "prompt_tokens": 137, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q11", "condition": "B", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 137, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q12", "condition": "A", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 7.34, "prompt_tokens": 595, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q12", "condition": "A", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 6.85, "prompt_tokens": 595, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q12", "condition": "B", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 7.07, "prompt_tokens": 115, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q12", "condition": "B", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 115, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q13", "condition": "A", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 7.09, "prompt_tokens": 192, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q13", "condition": "A", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 6.87, "prompt_tokens": 192, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q13", "condition": "B", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 7.02, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q13", "condition": "B", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q14", "condition": "A", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 7.55, "prompt_tokens": 741, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q14", "condition": "A", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 6.91, "prompt_tokens": 741, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q14", "condition": "B", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 7.11, "prompt_tokens": 120, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q14", "condition": "B", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 120, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q15", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 7.19, "prompt_tokens": 605, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q15", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 6.83, "prompt_tokens": 605, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q15", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 7.07, "prompt_tokens": 116, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q15", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 116, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q16", "condition": "A", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.21, "prompt_tokens": 429, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q16", "condition": "A", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 6.86, "prompt_tokens": 429, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q16", "condition": "B", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 7.08, "prompt_tokens": 118, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q16", "condition": "B", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 118, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q17", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 7.52, "prompt_tokens": 708, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q17", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 6.89, "prompt_tokens": 708, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q17", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 7.12, "prompt_tokens": 121, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q17", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 6.88, "prompt_tokens": 121, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q18", "condition": "A", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 7.43, "prompt_tokens": 648, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q18", "condition": "A", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 6.9, "prompt_tokens": 648, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q18", "condition": "B", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 7.12, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "M13-Qwen3.5-35B-A3B-UD-Q6_K", "question": "Q18", "condition": "B", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 6.85, "prompt_tokens": 119, "completion_tokens": 400, "answer": ""}
{"model": "gemma-7b-it-Q8_0", "question": "Q01", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 1.65, "prompt_tokens": 760, "completion_tokens": 49, "answer": "Данных по этому вопросу в базе знаний нет. Испытательный срок для обычного сотрудника и руководителей подразделений описан в фрагментах [Д01#1] и [Д01#1] соответственно."}
{"model": "gemma-7b-it-Q8_0", "question": "Q01", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 0.59, "prompt_tokens": 760, "completion_tokens": 31, "answer": "Данных по этому вопросу в базе знаний нет. Текст не содержит информацию о длине испытательного срока обычного сотрудника и руководителей."}
{"model": "gemma-7b-it-Q8_0", "question": "Q01", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 0.75, "prompt_tokens": 130, "completion_tokens": 31, "answer": "Данных по этому вопросу нет, так как я не могу ознакомиться с регламентами компании \"Вектор-Системы\"."}
{"model": "gemma-7b-it-Q8_0", "question": "Q01", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д05#2", "Д09#3", "Д05#3", "Д10#3"], "retrieval_s": 0.0, "latency_s": 0.58, "prompt_tokens": 130, "completion_tokens": 32, "answer": "Данных по этому вопросу нет, так как я не обладал информацией о регламентах компании «Вектор-Системы»."}
{"model": "gemma-7b-it-Q8_0", "question": "Q02", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 1.05, "prompt_tokens": 439, "completion_tokens": 34, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о частоте изменения пароля или минимальной длине пароля."}
{"model": "gemma-7b-it-Q8_0", "question": "Q02", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.59, "prompt_tokens": 439, "completion_tokens": 33, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о фактах, поэтому я не могу ответить на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q02", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.81, "prompt_tokens": 126, "completion_tokens": 39, "answer": "Данных по этому вопросу нет. Я не могу отвечать на этот вопрос, так как я не имеют информацию о регламентах компании \"Вектор-Системы\"."}
{"model": "gemma-7b-it-Q8_0", "question": "Q02", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д09#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.54, "prompt_tokens": 126, "completion_tokens": 30, "answer": "Данных по этому вопросу нет. Я не могу давать информацию о регламентах компании \"Вектор-Системы\"."}
{"model": "gemma-7b-it-Q8_0", "question": "Q03", "condition": "A", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 1.22, "prompt_tokens": 743, "completion_tokens": 34, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о суточных при командировке и лимите на гостиницу."}
{"model": "gemma-7b-it-Q8_0", "question": "Q03", "condition": "A", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 0.64, "prompt_tokens": 743, "completion_tokens": 34, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о суточных при командировке и лимите на гостиницу."}
{"model": "gemma-7b-it-Q8_0", "question": "Q03", "condition": "B", "run": 1, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 0.43, "prompt_tokens": 129, "completion_tokens": 18, "answer": "Данных по этому вопросу нет. Я не могу отвечать на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q03", "condition": "B", "run": 2, "retrieved": ["Д03#1", "Д01#3", "Д05#3", "Д07#1", "Д04#2"], "retrieval_s": 0.0, "latency_s": 0.69, "prompt_tokens": 129, "completion_tokens": 38, "answer": "Данных по этому вопросу нет. Я не могу предоставлять информацию о суточных при командировке и лимите на гостиницу компании «Вектор-Системы»."}
{"model": "gemma-7b-it-Q8_0", "question": "Q04", "condition": "A", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 1.42, "prompt_tokens": 764, "completion_tokens": 38, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о количестве дней в неделе, когда удалёнщик должен быть в офисе."}
{"model": "gemma-7b-it-Q8_0", "question": "Q04", "condition": "A", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 0.91, "prompt_tokens": 764, "completion_tokens": 48, "answer": "Данных по этому вопросу в базе знаний нет. \n\nФрагменты не содержат информацию о количестве дней в неделю, которые удалёнщик обязан быть в офисе, поэтому я не могу ответить на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q04", "condition": "B", "run": 1, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 0.78, "prompt_tokens": 135, "completion_tokens": 37, "answer": "Данных по этому вопросу нет. Я не могу отвечать на этот вопрос, так как я не располагаю информацией о компании \"Вектор-Системы\"."}
{"model": "gemma-7b-it-Q8_0", "question": "Q04", "condition": "B", "run": 2, "retrieved": ["Д04#1", "Д04#3", "Д01#3", "Д09#1", "Д08#3"], "retrieval_s": 0.0, "latency_s": 0.58, "prompt_tokens": 135, "completion_tokens": 32, "answer": "Данных по этому вопросу нет, так как я не располагаю информацией о регламентах компании «Вектор-Системы»."}
{"model": "gemma-7b-it-Q8_0", "question": "Q05", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.97, "prompt_tokens": 398, "completion_tokens": 31, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о количественном updates или подании заявки."}
{"model": "gemma-7b-it-Q8_0", "question": "Q05", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.69, "prompt_tokens": 398, "completion_tokens": 38, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о частоте обновления ноутбука и монитора, а также о подании заявки."}
{"model": "gemma-7b-it-Q8_0", "question": "Q05", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.42, "prompt_tokens": 131, "completion_tokens": 18, "answer": "Данных по этому вопросу нет. Я не могу отвечать на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q05", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д04#2", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.67, "prompt_tokens": 131, "completion_tokens": 37, "answer": "Данных по этому вопросу нет. Я не могу отвечать на вопросы о компании «Вектор-Системы», так как я не обладает информацией о ней."}
{"model": "gemma-7b-it-Q8_0", "question": "Q06", "condition": "A", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 1.18, "prompt_tokens": 689, "completion_tokens": 33, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о фактах, поэтому я не могу ответить на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q06", "condition": "A", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.63, "prompt_tokens": 689, "completion_tokens": 34, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о фактах, поэтому я не могу ответь на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q06", "condition": "B", "run": 1, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.94, "prompt_tokens": 128, "completion_tokens": 46, "answer": "Данных по этому вопросу нет. Я не могу предоставить информацию о време реакции и решении заявок Critical и Major, так как я не обладает данными о компании \"Вектор-Системы\"."}
{"model": "gemma-7b-it-Q8_0", "question": "Q06", "condition": "B", "run": 2, "retrieved": ["Д06#2", "Д06#3", "Д06#1", "Д05#2", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.73, "prompt_tokens": 128, "completion_tokens": 40, "answer": "Данных по этому вопросу нет. Я не располагаю информацией о регламентах компании \"Вектор-Системы\", поэтому я не могу ответь на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q07", "condition": "A", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 1.56, "prompt_tokens": 778, "completion_tokens": 46, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о стоимости тарифы \"Вектор-Офис Standard\" или скидках за годовую оплату."}
{"model": "gemma-7b-it-Q8_0", "question": "Q07", "condition": "A", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 0.6, "prompt_tokens": 778, "completion_tokens": 32, "answer": "Данных по этому вопросу в базе знаний нет. Исходя из приведённых фрагментов, я не могу ответь на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q07", "condition": "B", "run": 1, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 0.85, "prompt_tokens": 141, "completion_tokens": 41, "answer": "Данных по этому вопросу нет. Я не могу предоставить информацию о стоимости тариф \"Вектор-Офис Standard\" и скидках за годовую оплату."}
{"model": "gemma-7b-it-Q8_0", "question": "Q07", "condition": "B", "run": 2, "retrieved": ["Д08#1", "Д08#2", "Д08#0", "Д08#3", "Д06#1"], "retrieval_s": 0.0, "latency_s": 0.53, "prompt_tokens": 141, "completion_tokens": 29, "answer": "Данных по этому вопросу нет. Я не могу предоставлять информацию о тарифх компании \"Вектор-Системы\"."}
{"model": "gemma-7b-it-Q8_0", "question": "Q08", "condition": "A", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 1.31, "prompt_tokens": 680, "completion_tokens": 38, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о количестве дней ежегодного отпуска и о периоде подания заявки."}
{"model": "gemma-7b-it-Q8_0", "question": "Q08", "condition": "A", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 0.67, "prompt_tokens": 680, "completion_tokens": 36, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о количестве дней ежегодного отпуска или времени подания заявки."}
{"model": "gemma-7b-it-Q8_0", "question": "Q08", "condition": "B", "run": 1, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 0.61, "prompt_tokens": 129, "completion_tokens": 28, "answer": "Данных по этому вопросу нет, так как я не имеют доступ к информации о компании «Вектор-Системы»."}
{"model": "gemma-7b-it-Q8_0", "question": "Q08", "condition": "B", "run": 2, "retrieved": ["Д09#3", "Д09#2", "Д05#2", "Д09#1", "Д09#0"], "retrieval_s": 0.0, "latency_s": 0.71, "prompt_tokens": 129, "completion_tokens": 39, "answer": "Данных по этому вопросу нет, поэтому я не могу ответить. Я не имеет доступ к информации о компании \"Вектор-Системы\" и её регламентах."}
{"model": "gemma-7b-it-Q8_0", "question": "Q09", "condition": "A", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 1.25, "prompt_tokens": 701, "completion_tokens": 31, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о получении второго монитора для клиентских проектов."}
{"model": "gemma-7b-it-Q8_0", "question": "Q09", "condition": "A", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.57, "prompt_tokens": 701, "completion_tokens": 31, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о получении второго монитора для клиентских проектов."}
{"model": "gemma-7b-it-Q8_0", "question": "Q09", "condition": "B", "run": 1, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.78, "prompt_tokens": 135, "completion_tokens": 37, "answer": "Данных по этому вопросу нет. Удалёнщик должен обращаться к менеджеру проекта, чтобы получить информацию о возможностях получения второго монитора для клиентских проектов."}
{"model": "gemma-7b-it-Q8_0", "question": "Q09", "condition": "B", "run": 2, "retrieved": ["Д05#1", "Д05#2", "Д04#2", "Д02#3", "Д08#0"], "retrieval_s": 0.0, "latency_s": 0.78, "prompt_tokens": 135, "completion_tokens": 43, "answer": "Данных по этому вопросу нет. Я не могу предоставить информацию о регламентах компании \"Вектор-Системы\" по получению дополнительного монитора для клиентских проектов."}
{"model": "gemma-7b-it-Q8_0", "question": "Q10", "condition": "A", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 1.38, "prompt_tokens": 769, "completion_tokens": 36, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о стоимости гостеницы и времени сдачи авансового отчета."}
{"model": "gemma-7b-it-Q8_0", "question": "Q10", "condition": "A", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 0.71, "prompt_tokens": 769, "completion_tokens": 38, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о стоимости гостеприимства или времени сдачи авансового отчета."}
{"model": "gemma-7b-it-Q8_0", "question": "Q10", "condition": "B", "run": 1, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 1.02, "prompt_tokens": 158, "completion_tokens": 48, "answer": "Данных по этому вопросу нет. Я не могу ответь, когда подавать заявку в СЭД \"Дирекум\", сколько можно тратить на гостиницу и когда сдавать авансовый отчёт."}
{"model": "gemma-7b-it-Q8_0", "question": "Q10", "condition": "B", "run": 2, "retrieved": ["Д04#3", "Д03#3", "Д10#0", "Д03#2", "Д01#3"], "retrieval_s": 0.0, "latency_s": 0.76, "prompt_tokens": 158, "completion_tokens": 42, "answer": "Данных по этому вопросу нет. Я не могу отвечать на этот вопрос, так как у меня нет информации о компании \"Вектор-Системы\" и ее регламентах."}
{"model": "gemma-7b-it-Q8_0", "question": "Q11", "condition": "A", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 1.16, "prompt_tokens": 833, "completion_tokens": 23, "answer": "Данных по этому вопросу в базе знаний нет. therefore, я не могу ответь на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q11", "condition": "A", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 0.36, "prompt_tokens": 833, "completion_tokens": 19, "answer": "Данных по этому вопросу в базе знаний нет, поэтому я не могу отвечать."}
{"model": "gemma-7b-it-Q8_0", "question": "Q11", "condition": "B", "run": 1, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 1.06, "prompt_tokens": 149, "completion_tokens": 52, "answer": "Данных по этому вопросу нет. Я не могу предоставить информацию о длинии цикла согласования договора и стоимости со скидкой за год, так как я не располагаю данными о компании \"Вектор-Системы\"."}
{"model": "gemma-7b-it-Q8_0", "question": "Q11", "condition": "B", "run": 2, "retrieved": ["Д10#2", "Д03#2", "Д01#2", "Д09#2", "Д08#1"], "retrieval_s": 0.0, "latency_s": 1.04, "prompt_tokens": 149, "completion_tokens": 57, "answer": "Данных по этому вопросу нет. Я не могу предоставить информацию о длительности цикла согласования договора и стоимости тариф Enterprise с годовой оплатой, так как я не располагаю о всех деталях о компании \"Вектор-Системы\"."}
{"model": "gemma-7b-it-Q8_0", "question": "Q12", "condition": "A", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 1.07, "prompt_tokens": 605, "completion_tokens": 26, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о годовой премии сотрудников."}
{"model": "gemma-7b-it-Q8_0", "question": "Q12", "condition": "A", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.35, "prompt_tokens": 605, "completion_tokens": 19, "answer": "Данных по этому вопросу в базе знаний нет, поэтому я не могу ответь."}
{"model": "gemma-7b-it-Q8_0", "question": "Q12", "condition": "B", "run": 1, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.44, "prompt_tokens": 123, "completion_tokens": 18, "answer": "Данных по этому вопросу нет. Я не могу ответь на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q12", "condition": "B", "run": 2, "retrieved": ["Д08#2", "Д01#0", "Д03#0", "Д05#0", "Д06#0"], "retrieval_s": 0.0, "latency_s": 0.33, "prompt_tokens": 123, "completion_tokens": 18, "answer": "Данных по этому вопросу нет. Я не могу отвечать на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q13", "condition": "A", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.47, "prompt_tokens": 195, "completion_tokens": 21, "answer": "Данных по этому вопросу в базе знаний нет. therefore, я не могу ответить на него."}
{"model": "gemma-7b-it-Q8_0", "question": "Q13", "condition": "A", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.49, "prompt_tokens": 195, "completion_tokens": 27, "answer": "Данных по этому вопросу в базе знаний нет. Ответ: «Данных по этому вопросу в базе знаний нет»."}
{"model": "gemma-7b-it-Q8_0", "question": "Q13", "condition": "B", "run": 1, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.56, "prompt_tokens": 126, "completion_tokens": 26, "answer": "Данных по этому вопросу нет. Я не могу предоставлять информацию о процедуре оформления корпоративной SIM-карты."}
{"model": "gemma-7b-it-Q8_0", "question": "Q13", "condition": "B", "run": 2, "retrieved": ["Д01#0", "Д01#1", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.67, "prompt_tokens": 126, "completion_tokens": 37, "answer": "Данных по этому вопросу нет, так как я не могу доступ к конфиденциальной информации, например, как procedures компании \"Вектор-Системы\"."}
{"model": "gemma-7b-it-Q8_0", "question": "Q14", "condition": "A", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.92, "prompt_tokens": 749, "completion_tokens": 18, "answer": "Данных по этому вопросу в базе знаний нет, поэтому я не могу ответить."}
{"model": "gemma-7b-it-Q8_0", "question": "Q14", "condition": "A", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.62, "prompt_tokens": 749, "completion_tokens": 33, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о тарифе \"Вектор-Офис Premium\"."}
{"model": "gemma-7b-it-Q8_0", "question": "Q14", "condition": "B", "run": 1, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.45, "prompt_tokens": 129, "completion_tokens": 19, "answer": "Данных по этому вопросу нет, поэтому я не могу ответь на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q14", "condition": "B", "run": 2, "retrieved": ["Д08#0", "Д08#1", "Д08#3", "Д08#2", "Д03#0"], "retrieval_s": 0.0, "latency_s": 0.33, "prompt_tokens": 129, "completion_tokens": 18, "answer": "Данных по этому вопросу нет, поэтому я не могу ответь на него."}
{"model": "gemma-7b-it-Q8_0", "question": "Q15", "condition": "A", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.74, "prompt_tokens": 616, "completion_tokens": 20, "answer": "Данных по этому вопросу в базе знаний нет, поэтому я не могу ответить на него."}
{"model": "gemma-7b-it-Q8_0", "question": "Q15", "condition": "A", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.48, "prompt_tokens": 616, "completion_tokens": 26, "answer": "Данных по этому вопросу в базе знаний нет. Текст не содержит информацию о процедуре оформления материнского капитала."}
{"model": "gemma-7b-it-Q8_0", "question": "Q15", "condition": "B", "run": 1, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.6, "prompt_tokens": 125, "completion_tokens": 27, "answer": "Данных по этому вопросу нет, так как я не обладает информацией о компании \"Вектор-Системы\"."}
{"model": "gemma-7b-it-Q8_0", "question": "Q15", "condition": "B", "run": 2, "retrieved": ["Д01#1", "Д01#0", "Д01#2", "Д01#3", "Д02#0"], "retrieval_s": 0.0, "latency_s": 0.78, "prompt_tokens": 125, "completion_tokens": 43, "answer": "Данных по этому вопросу нет. Я не могу описать процедуру оформления материнского капитала у компании \"Вектор-Системы\", так как я не располагаю информацией о ней."}
{"model": "gemma-7b-it-Q8_0", "question": "Q16", "condition": "A", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.82, "prompt_tokens": 436, "completion_tokens": 26, "answer": "Данных по этому вопросу в базе знаний нет. Текст не содержит информацию о столовой или menu в офисе."}
{"model": "gemma-7b-it-Q8_0", "question": "Q16", "condition": "A", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.5, "prompt_tokens": 436, "completion_tokens": 27, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о столовой или menuе."}
{"model": "gemma-7b-it-Q8_0", "question": "Q16", "condition": "B", "run": 1, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.59, "prompt_tokens": 127, "completion_tokens": 27, "answer": "Данных по этому вопросу нет. Я не могу предоставить информацию о столовоей иmenu в офисе компании."}
{"model": "gemma-7b-it-Q8_0", "question": "Q16", "condition": "B", "run": 2, "retrieved": ["Д10#1", "Д04#1", "Д01#0", "Д01#1", "Д01#2"], "retrieval_s": 0.0, "latency_s": 0.66, "prompt_tokens": 127, "completion_tokens": 36, "answer": "Данных по этому вопросу нет. Я не располагаю информацией о компании \"Вектор-Системы\", поэтому я не могу ответь на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q17", "condition": "A", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 1.11, "prompt_tokens": 728, "completion_tokens": 27, "answer": "Данных по этому вопросу в базе знаний нет. Фрагменты не содержат информацию о частоте изменения пароля."}
{"model": "gemma-7b-it-Q8_0", "question": "Q17", "condition": "A", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 0.45, "prompt_tokens": 728, "completion_tokens": 24, "answer": "Данных по этому вопросу в базе знаний нет. Текст не содержит информацию о частоте изменения пароля."}
{"model": "gemma-7b-it-Q8_0", "question": "Q17", "condition": "B", "run": 1, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 0.85, "prompt_tokens": 130, "completion_tokens": 41, "answer": "Данных по этому вопросу нет. Я не могу ответь на этот вопрос, так как я не располагаю информацией о регламентах компании «Вектор-Системы»."}
{"model": "gemma-7b-it-Q8_0", "question": "Q17", "condition": "B", "run": 2, "retrieved": ["Д02#1", "Д05#1", "Д09#1", "Д08#3", "Д03#2"], "retrieval_s": 0.0, "latency_s": 0.67, "prompt_tokens": 130, "completion_tokens": 37, "answer": "Данных по этому вопросу нет. Я не могу ответь на вопросы, касающиеся конкретных регламентов компании, так как у меня нет доступа к личным данным."}
{"model": "gemma-7b-it-Q8_0", "question": "Q18", "condition": "A", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 1.15, "prompt_tokens": 665, "completion_tokens": 29, "answer": "Данных по этому вопросу в базе знаний нет. Текст не содержит информации о возможности разделки отпуска на однодневные части."}
{"model": "gemma-7b-it-Q8_0", "question": "Q18", "condition": "A", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.54, "prompt_tokens": 665, "completion_tokens": 29, "answer": "Данных по этому вопросу в базе знаний нет. Текст не содержит информацию, которая бы позволила ответь на этот вопрос."}
{"model": "gemma-7b-it-Q8_0", "question": "Q18", "condition": "B", "run": 1, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.61, "prompt_tokens": 128, "completion_tokens": 28, "answer": "Данных по этому вопросу нет. Я не могу ответь, можно ли разбить весь отпуск на однодневные части."}
{"model": "gemma-7b-it-Q8_0", "question": "Q18", "condition": "B", "run": 2, "retrieved": ["Д09#1", "Д07#1", "Д01#3", "Д07#0", "Д04#0"], "retrieval_s": 0.0, "latency_s": 0.71, "prompt_tokens": 128, "completion_tokens": 39, "answer": "Данных по этому вопросу нет. Я не могу отвечать на этот вопрос, так как я не имеет информацию о регламентах компании \"Вектор-Системы\"."}"""

import json

RESULTS = [json.loads(_l) for _l in ANSWERS_RAW.splitlines() if _l.strip()]
print(f"Записей: {len(RESULTS)}; моделей: {len({r['model'] for r in RESULTS})}; "
      f"вопросов: {len({r['question'] for r in RESULTS})}; "
      f"условий: {sorted({r['condition'] for r in RESULTS})}")

## 1. Корпус знаний «Вектор-Системы»

Все 10 документов (партия 1) в полном объёме.

In [ ]:
from IPython.display import display, Markdown

for _name in sorted(CORPUS):
    display(Markdown(f"**{_name}**"))
    display(Markdown(CORPUS[_name]))
    display(Markdown("---"))

## 2. Банк вопросов (18)

`gold` — золотые документы (для recall); `—` у «ловушек» означает, что ответа в корпусе нет.

In [ ]:
import pandas as pd

qdf = pd.DataFrame(QUESTIONS, columns=["id", "type", "gold", "question", "reference"])
qdf.columns = ["ID", "Тип", "Золотые документы", "Вопрос", "Эталонный ответ"]

def _scroll(df, max_height=600, max_width=None):
    html = df.to_html(escape=True, index=False, border=0)
    style = f"max-height:{max_height}px; overflow:auto;"
    if max_width:
        style += f"max-width:{max_width}px;"
    from IPython.display import display, HTML
    display(HTML(f'<div style="{style}">{html}</div>'))

_scroll(qdf, max_height=500)

## 3. Ответы моделей (720 записей)

Полные сырые записи, без агрегаций. Поля:

| Поле | Смысл |
|---|---|
| `model` | модель (GGUF-квант для локальных) |
| `question` | ID вопроса (Q01–Q18) |
| `condition` | A — с контекстом (5 фрагментов), B — без контекста |
| `run` | номер прогона (1–2) |
| `retrieved` | отретриванные фрагменты (BM25, top-5), порядок как в промпте |
| `retrieval_s` | время поиска, с |
| `latency_s` | время ответа модели, с |
| `prompt_tokens` / `completion_tokens` | токены запроса / ответа |
| `answer` | полный текст ответа модели |

In [ ]:
rdf = pd.DataFrame(RESULTS)
rdf["retrieved"] = rdf["retrieved"].apply(", ".join)
rdf = rdf[["model", "question", "condition", "run", "retrieved", "retrieval_s",
           "latency_s", "prompt_tokens", "completion_tokens", "answer"]]

from IPython.display import display, Markdown

for _c in ["A", "B"]:
    _label = {"A": "Условие A — с отретриванным контекстом (360 записей)",
              "B": "Условие B — без контекста (360 записей)"}[_c]
    display(Markdown(f"### {_label}"))
    _scroll(rdf[rdf["condition"] == _c], max_height=600)